In [1]:
import os

print("현재 작업 디렉터리:")
print(os.getcwd())

현재 작업 디렉터리:
/home/yjlee/Research/oa-orthogonal-platform


In [2]:
#핵심 파이프라인 + Q3/Q4 진단 + OA-Colour 후속분석과의 연결#
#Q1제거, Q5탐색안함, 

In [3]:
"""
==============================================================================
Open Access as Orthogonal Platform — Unified Core Pipeline
(Core build + Q3 sample-window + Q4 OA-colour diagnostics)
------------------------------------------------------------------------------
Scope decisions per author instruction:
  - Q1 (Table A5, zero-citation robustness)  -> EXCLUDED entirely
  - Q3 (sample window: extraction/analysis)  -> INCLUDED
  - Q4 (gold/green/bronze OA colour share)   -> INCLUDED
  - Q5 (C14 COVID/SARS cluster pre-2013 OA)  -> NOT searched / EXCLUDED

This script also serves as the REQUIRED upstream step for
`oa_colour_heterogeneity_pipeline.py`: the final saved CSV
(outputs/tables/analysis_final.csv) now includes every column that
pipeline's REQUIRED_COLS expects (OA_Colour, recent, log_field_div,
cluster, pre_field_oa, primary_field, log_patent, log_citation,
oa_dummy, Pub_Year), which the original save_cols list omitted.
==============================================================================
"""

# -- Standard library --------------------------------------------------------
import os
import re
import math
import warnings
import time
import json
import functools
import argparse
from pathlib import Path
from collections import Counter
from typing import Optional

os.environ["TF_CPP_MIN_LOG_LEVEL"]    = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"]   = "0"
os.environ["TF_KERAS_LEGACY_LOGGING"] = "0"

# -- Third-party --------------------------------------------------------------
import numpy as np
import pandas as pd
import scipy.stats as ss

warnings.filterwarnings("ignore")

import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.regression.quantile_regression import QuantReg
from linearmodels.iv import IV2SLS

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# torch / sentence-transformers only needed for clustering (STEP 3).
try:
    import torch
    torch.set_num_threads(os.cpu_count() or 4)
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("  WARNING: torch not found -> clustering step will be unavailable")

try:
    from sentence_transformers import SentenceTransformer
    HAS_SENTENCE_TRANSFORMERS = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False
    print("  WARNING: sentence-transformers not found -> clustering step will be unavailable")


# ==============================================================================
# UMAP / sklearn compatibility patch (scoped ONLY to umap's own namespace)
# ==============================================================================
def _apply_sklearn_umap_patch():
    try:
        import umap.umap_ as _umap_mod
    except ImportError:
        return
    except Exception as e:
        print(f"  [PATCH] Skipped ({e})")
        return

    import sklearn.utils.validation as _skval
    _orig = _skval.check_array
    try:
        import inspect
        _accepted_params = set(inspect.signature(_orig).parameters)
    except (TypeError, ValueError):
        _accepted_params = set()

    @functools.wraps(_orig)
    def _patched(*args, **kwargs):
        if "force_all_finite" in kwargs and "force_all_finite" not in _accepted_params:
            kwargs["ensure_all_finite"] = kwargs.pop("force_all_finite")
        elif "ensure_all_finite" in kwargs and "ensure_all_finite" not in _accepted_params:
            kwargs["force_all_finite"] = kwargs.pop("ensure_all_finite")
        return _orig(*args, **kwargs)

    _umap_mod.check_array = _patched
    print("  [PATCH] umap-learn/scikit-learn keyword compatibility patch applied "
          "(scoped to umap only)")


_apply_sklearn_umap_patch()

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("  WARNING: umap-learn not found -> PCA fallback")

try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("  WARNING: hdbscan not found -> KMeans fallback")

try:
    from econml.dml import CausalForestDML
    HAS_ECONML = True
except ImportError:
    HAS_ECONML = False
    print("  WARNING: econml not found -> DR-Learner fallback")


# ==============================================================================
# CONFIG
# ==============================================================================
CSV_PATH = os.environ.get(
    "OA_PIPELINE_CSV_PATH",
    "/home/yjlee/Research/oa-orthogonal-platform/Transport_CN_Scholarly_Works.csv",
)
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
for _sub in ["01_descriptive", "02_main_results", "03_platform_framing",
             "04_causal_identification", "05_heterogeneity", "06_robustness",
             "07_combined"]:
    (OUT_DIR / "figures" / _sub).mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE   = 64
N_CLUSTERS   = 8
SEED         = 2025
POLICY_YEAR  = 2015
PLACEBO_YEAR = 2012
TOP_PCT      = 0.99
ES_WIN_PRE   = -2
ES_WIN_POST  = +3

np.random.seed(SEED)

C_BLUE, C_RED, C_GREY   = "#1D4E89", "#C0392B", "#7F8C8D"
C_GREEN, C_ORANGE, C_PURPLE = "#1A7A4A", "#E67E22", "#7D3C98"
C_TEAL = "#148F77"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.35, "grid.linewidth": 0.7,
    "legend.framealpha": 0.9, "legend.fontsize": 9.5,
    "figure.dpi": 150, "savefig.dpi": 200, "savefig.bbox": "tight",
})
sns.set_palette("muted")

CN_INST_LIST = {
    "Zhejiang University", "Xi'an Jiaotong University",
    "Wuhan University of Technology", "Wuhan University",
    "University of Science and Technology of China",
    "University of Chinese Academy of Sciences", "Tsinghua University",
    "Tongji University", "Tianjin University", "Sun Yat-sen University",
    "Southwest Jiaotong University", "Southeast University",
    "South China University of Technology", "Soochow University (Suzhou)",
    "Sichuan University", "Shenzhen University", "Shanghai University",
    "Shanghai Jiao Tong University", "Shandong University", "Peking University",
    "Nanjing University", "Jilin University",
    "Huazhong University of Science and Technology",
    "Harbin Institute of Technology", "Fudan University",
    "Dalian University of Technology", "Chongqing University",
    "Chinese Academy of Sciences", "Central South University",
    "Beijing University of Technology", "Beijing Jiaotong University",
    "Beijing Institute of Technology", "Beihang University",
}
CN_KEYWORDS = [
    "tsinghua", "peking", "fudan", "zhejiang", "tongji", "wuhan", "harbin",
    "xian jiaotong", "xi'an jiaotong", "southeast university", "sun yat-sen",
    "zhongshan", "nankai", "tianjin", "dalian", "chongqing", "sichuan",
    "jilin", "central south", "south china", "huazhong", "nanjing",
    "northeastern", "southwest jiaotong", "chinese academy",
    "shenzhen university", "beijing jiaotong", "beijing institute",
    "beihang", "soochow", "chang'an", "changan", "lanzhou", "hefei",
    "university of science and technology of china",
]


def is_cn(institution: str) -> bool:
    if not isinstance(institution, str) or not institution.strip():
        return False
    if institution.strip() in CN_INST_LIST:
        return True
    low = institution.lower()
    return any(kw in low for kw in CN_KEYWORDS)


# ==============================================================================
# Utilities
# ==============================================================================
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.bool_):    return bool(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)


def _parse_bool_series(series) -> pd.Series:
    if series is None:
        return pd.Series(0, index=range(0), dtype=int)
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(int)
    if pd.api.types.is_integer_dtype(series) or pd.api.types.is_float_dtype(series):
        return series.fillna(0).astype(int)
    TRUE_SET = {"true", "1", "yes", "y", "t", "open", "oa"}

    def _map(v):
        if pd.isna(v):
            return 0
        sv = str(v).strip().lower()
        if sv in TRUE_SET:
            return 1
        try:
            return int(float(sv) != 0)
        except Exception:
            return 0

    return series.map(_map).fillna(0).astype(int)


def _clean_df(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    return df.replace([np.inf, -np.inf], np.nan).dropna(subset=cols).copy().reset_index(drop=True)


def _get_param(m, var):
    if m is None: return np.nan
    try:
        return float(m.params[var])
    except Exception:
        return np.nan


def _get_pval(m, var):
    if m is None: return np.nan
    try:
        return float(m.pvalues[var])
    except Exception:
        return np.nan


# ==============================================================================
# Row-integrity QC (delimiter/column-shift detector)
# ==============================================================================
def qc_row_integrity(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    n_before = len(df)

    empty_mask = df.isnull().all(axis=1)
    n_empty = int(empty_mask.sum())
    df = df[~empty_mask].copy()

    shifted_mask = pd.Series(False, index=df.index)

    if "Is Open Access" in df.columns:
        valid_oa_vals = {"TRUE", "FALSE", True, False}
        bad_oa = ~df["Is Open Access"].isin(valid_oa_vals) & df["Is Open Access"].notna()
        shifted_mask |= bad_oa

    if "Publication Year" in df.columns:
        yr_numeric = pd.to_numeric(df["Publication Year"], errors="coerce")
        bad_yr = yr_numeric.isna() & df["Publication Year"].notna()
        shifted_mask |= bad_yr

    n_shifted = int(shifted_mask.sum())

    print("  [QC] Row Integrity Check")
    print(f"    Fully-empty rows dropped: {n_empty:,}")
    print(f"    Column-shifted rows flagged & dropped: {n_shifted:,}")
    if n_shifted > 0 and "Lens ID" in df.columns:
        print(f"    Example shifted Lens IDs: "
              f"{df.loc[shifted_mask, 'Lens ID'].head(5).tolist()}")
        print("    Recommendation: re-export these Lens IDs from Lens.org directly "
              "rather than attempting automated column-shift repair.")

    df_clean = df[~shifted_mask].copy()
    print(f"    Rows: {n_before:,} -> {len(df_clean):,}")
    return df_clean


def _read_csv_robust(path: str) -> pd.DataFrame:
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr", "latin-1"]:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            print(f"  Encoding: [{enc}]")
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    return pd.read_csv(path, encoding="utf-8", errors="replace", low_memory=False)


# ==============================================================================
# STEP 1 — Data loading (single analysis sample: CN + Citing_Patents >= 1)
# ==============================================================================
def load_data_raw_cn(path: str, run_qc: bool = True) -> pd.DataFrame:
    """
    Returns the China-affiliated, patent-cited (>=1) analysis sample
    BEFORE build_indices() -- i.e. raw feature columns only. Q3 needs this
    raw form (to compare extraction-year N against post-build analysis-year N).
    """
    print("\n" + "=" * 65)
    print("[STEP 1] Data Loading + Preprocessing")
    print("=" * 65)

    df = _read_csv_robust(path)
    df.columns = df.columns.str.strip()

    if run_qc:
        df = qc_row_integrity(df)

    df["Citing_Patents"] = pd.to_numeric(df.get("Citing Patents Count"), errors="coerce")
    df["Citing_Works"]   = pd.to_numeric(df.get("Citing Works Count"),   errors="coerce")
    df["Pub_Year"]       = pd.to_numeric(df.get("Publication Year"),     errors="coerce")

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df[(df["Pub_Year"] >= 1900) & (df["Pub_Year"] <= 2025)]
    df = df.dropna(subset=["Pub_Year", "Citing_Patents"]).copy()

    df["OA"]          = _parse_bool_series(df.get("Is Open Access")).fillna(0).astype(int)
    df["Pub_Type"]     = df.get("Publication Type", pd.Series(dtype=str)).fillna("unknown").astype(str).str.lower()
    df["Institution"]  = df.get("Institution", pd.Series("", index=df.index)).fillna("").astype(str)

    if "Open Access Colour" in df.columns:
        df["OA_Colour"] = df["Open Access Colour"].fillna("closed/unknown").astype(str)
    else:
        df["OA_Colour"] = "unknown"

    if "Source Country" in df.columns:
        df["Source_Country"] = df["Source Country"].fillna("").astype(str)
        df = df[~df["Source_Country"].str.contains(";", na=False)]
    else:
        df["Source_Country"] = ""

    df["is_CN"] = df["Institution"].apply(is_cn)
    df.loc[df["Source_Country"].str.strip() == "China", "is_CN"] = True

    df_cn = df[df["is_CN"]].copy().reset_index(drop=True)
    n_cn = len(df_cn)

    # Analysis sample = patent-cited only (Q1 zero-citation robustness excluded
    # from scope, so we no longer keep a separate all-citation copy)
    df_cn_analysis = df_cn[df_cn["Citing_Patents"] >= 1].copy().reset_index(drop=True)

    print(f"  Total -> CN -> Patent>=1: {len(df):,} -> {n_cn:,} -> {len(df_cn_analysis):,}")
    return df_cn_analysis


# ==============================================================================
# STEP 2 — Feature engineering
# ==============================================================================
def build_indices(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["log_patent"]   = np.log1p(df["Citing_Patents"])
    df["log_citation"] = np.log1p(df["Citing_Works"])
    df["oa_dummy"]     = df["OA"].astype(int)
    df["recent"]       = (df["Pub_Year"] >= 2018).astype(int)
    df["post_policy"]  = (df["Pub_Year"] >= POLICY_YEAR).astype(int)
    df["post_placebo"] = (df["Pub_Year"] >= PLACEBO_YEAR).astype(int)
    df["year_rel"]     = (df["Pub_Year"] - POLICY_YEAR).astype(int)

    thr = df["Citing_Patents"].quantile(TOP_PCT)
    df["top1pct"] = (df["Citing_Patents"] >= thr).astype(int)

    def parse_primary_field(s):
        if not isinstance(s, str) or not s.strip(): return "Unknown"
        parts = [x.strip() for x in re.split(r"[;,]", s) if x.strip()]
        return parts[0] if parts else "Unknown"

    def field_count(s):
        if not isinstance(s, str) or not s.strip(): return 1
        return max(1, len([x for x in re.split(r"[;,]", s) if x.strip()]))

    def interdiscipl_entropy(s):
        if not isinstance(s, str) or not s.strip(): return 0.0
        parts = [x.strip() for x in re.split(r"[;,]", s) if x.strip()]
        return math.log(len(parts)) if len(parts) > 1 else 0.0

    fos = df.get("Fields of Study", pd.Series("", index=df.index))
    df["primary_field"]       = fos.apply(parse_primary_field)
    df["field_diversity"]     = fos.apply(field_count)
    df["log_field_div"]       = np.log1p(df["field_diversity"])
    df["interdisciplinarity"] = fos.apply(interdiscipl_entropy)

    oa_rate_yr = df.groupby("Pub_Year")["oa_dummy"].mean().rename("oa_rate_global_yr")
    df = df.merge(oa_rate_yr, on="Pub_Year", how="left")

    pre_fld = (df[df["Pub_Year"] < POLICY_YEAR]
               .groupby("primary_field")["oa_dummy"]
               .mean().rename("pre_field_oa").reset_index())
    df = df.merge(pre_fld, on="primary_field", how="left")
    df["pre_field_oa"] = df["pre_field_oa"].fillna(df["oa_dummy"].mean())

    df["iv_shift_share"] = df["oa_rate_global_yr"].fillna(0) * df["pre_field_oa"].fillna(0)

    iv_raw = df["oa_rate_global_yr"].fillna(0) * df["pre_field_oa"].fillna(0)
    field_mean_iv = df["primary_field"].map(
        pd.Series(iv_raw.values, index=df.index).groupby(df["primary_field"]).mean())
    iv_field_resid = iv_raw - field_mean_iv
    year_mean_iv = df["Pub_Year"].map(
        pd.Series(iv_field_resid.values, index=df.index).groupby(df["Pub_Year"]).mean())
    iv_field_year_resid = iv_field_resid - year_mean_iv
    tmp = pd.DataFrame({
        "z_raw": iv_field_year_resid.values,
        "log_citation": df["log_citation"].values,
        "log_field_div": df["log_field_div"].values,
        "pre_field_oa": df["pre_field_oa"].values,
    }).replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) > 200:
        try:
            proj = OLS(tmp["z_raw"], sm.add_constant(tmp[["log_citation", "log_field_div", "pre_field_oa"]])).fit()
            resid_full = np.full(len(df), np.nan)
            resid_full[tmp.index] = proj.resid
            df["iv_residualized"] = resid_full
        except Exception:
            df["iv_residualized"] = iv_field_year_resid.values
    else:
        df["iv_residualized"] = iv_field_year_resid.values
    df["iv_residualized"] = df["iv_residualized"].fillna(0.0)

    global_oa_lag2 = df.groupby("Pub_Year")["oa_dummy"].mean().shift(2).rename("oa_rate_global_lag2")
    df = df.merge(global_oa_lag2.reset_index(), on="Pub_Year", how="left")
    df["iv_alt"] = df["oa_rate_global_lag2"].fillna(df["oa_rate_global_yr"]) * df["pre_field_oa"].fillna(0)

    raw_trend = df.groupby("primary_field")["Pub_Year"].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-6))
    df["field_yr_trend"] = raw_trend.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    fld_yr_oa = (df.groupby(["primary_field", "Pub_Year"])["oa_dummy"]
                   .mean().rename("field_oa_rate_yr").reset_index())
    df = df.merge(fld_yr_oa, on=["primary_field", "Pub_Year"], how="left")

    cohort_map = {}
    for fld, grp in df.groupby("primary_field"):
        g = grp.sort_values("Pub_Year")
        adopt = g[g["field_oa_rate_yr"] > 0.30]["Pub_Year"].min()
        cohort_map[fld] = int(adopt) if not pd.isna(adopt) else 9999
    df["cohort_year"] = df["primary_field"].map(cohort_map).fillna(9999).astype(int)

    med_exposure = df["pre_field_oa"].median()
    df["high_exposure"] = (df["pre_field_oa"] > med_exposure).astype(int)

    df["ddd_cont"]     = df["oa_dummy"] * df["post_policy"] * df["pre_field_oa"]
    df["ddd"]          = df["oa_dummy"] * df["post_policy"] * df["high_exposure"]
    df["oa_x_post"]    = df["oa_dummy"]   * df["post_policy"]
    df["oa_x_preoa"]   = df["oa_dummy"]   * df["pre_field_oa"]
    df["post_x_preoa"] = df["post_policy"] * df["pre_field_oa"]

    df["disruptiveness"] = (
        df["Citing_Patents"] / (df["Citing_Patents"] + df["Citing_Works"].fillna(0) + 1))

    titles = df.get("Title", pd.Series("", index=df.index)).fillna("").astype(str).tolist()
    all_words = Counter()
    for t in titles:
        all_words.update(set(re.findall(r"\b[a-z]{4,}\b", t.lower())))
    n_docs = len(titles)
    rare_threshold = max(1, int(n_docs * 0.05))

    def novelty_score(title):
        words = set(re.findall(r"\b[a-z]{4,}\b", str(title).lower()))
        if not words: return 0.0
        rare = sum(1 for w in words if all_words.get(w, 0) <= rare_threshold)
        return rare / len(words)

    df["novelty_score"] = pd.Series(titles).apply(novelty_score).values

    df["text_combined"] = (
        df.get("Title",           pd.Series("", index=df.index)).fillna("") + " " +
        df.get("Abstract",        pd.Series("", index=df.index)).fillna("") + " " +
        df.get("Fields of Study", pd.Series("", index=df.index)).fillna("") + " " +
        df.get("Keywords",        pd.Series("", index=df.index)).fillna("")
    ).str.strip()

    df["log_oa_x_citation"]     = df["oa_dummy"] * df["log_citation"]
    df["pre_oa_x_recent"]       = df["pre_field_oa"] * df["recent"]
    df["citation_sq"]           = df["log_citation"] ** 2
    df["patent_citation_ratio"] = np.log1p(df["Citing_Patents"] / (df["Citing_Works"].fillna(1) + 1))

    required = ["log_patent", "log_citation", "oa_dummy", "Pub_Year",
                "iv_shift_share", "iv_residualized", "pre_field_oa", "field_yr_trend"]
    df = df.replace([np.inf, -np.inf], np.nan)
    df[required] = df[required].fillna(df[required].median())
    df = df.dropna(subset=["log_patent", "log_citation", "oa_dummy", "Pub_Year"]).copy()
    df = df.reset_index(drop=True)

    print(f"  Build complete: {len(df):,} rows x {df.shape[1]} cols")
    print(f"  OA rate: {df['oa_dummy'].mean()*100:.1f}%  |  "
          f"IV mean: {df['iv_shift_share'].mean():.3f}  |  "
          f"IV-Resid mean: {df['iv_residualized'].mean():.4f}  "
          f"std: {df['iv_residualized'].std():.4f}")
    return df


# ==============================================================================
# STEP 3 — Embedding + UMAP + HDBSCAN
# ==============================================================================
def embed_and_cluster(df: pd.DataFrame):
    print("\n[STEP 3] Sentence Embedding + UMAP + HDBSCAN")
    if not (HAS_TORCH and HAS_SENTENCE_TRANSFORMERS):
        raise ImportError(
            "embed_and_cluster() requires torch and sentence-transformers. "
            "Install with: pip install torch sentence-transformers --break-system-packages"
        )
    texts = df["text_combined"].tolist()
    texts = [t if t.strip() else "transport china research" for t in texts]

    model = SentenceTransformer(EMBED_MODEL)
    t0 = time.perf_counter()
    emb = model.encode(texts, batch_size=BATCH_SIZE,
                        show_progress_bar=True, normalize_embeddings=True,
                        convert_to_numpy=True)
    print(f"  Embedding: {emb.shape}  ({time.perf_counter()-t0:.1f}s)")
    del model

    pca_dim = min(50, emb.shape[1], emb.shape[0] - 1)
    emb_pca = PCA(n_components=pca_dim, random_state=SEED).fit_transform(emb).astype(np.float32)

    if HAS_UMAP:
        try:
            reducer = umap.UMAP(n_components=15, n_neighbors=15, min_dist=0.1,
                                 metric="cosine", random_state=SEED, low_memory=True)
            emb_15d = reducer.fit_transform(emb_pca)
            reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.05,
                                    metric="cosine", random_state=SEED, low_memory=True)
            emb_2d = reducer_2d.fit_transform(emb_pca)
        except Exception as e:
            print(f"  WARNING: UMAP failed: {e} -> PCA fallback")
            emb_15d = emb_pca
            emb_2d = PCA(n_components=2, random_state=SEED).fit_transform(emb_pca)
    else:
        emb_15d = emb_pca
        emb_2d = PCA(n_components=2, random_state=SEED).fit_transform(emb_pca)

    cluster_input = emb_15d.astype(np.float64)

    if HAS_HDBSCAN:
        best_labels, best_sil = None, -1.0
        for mcs in [max(30, len(df)//200), max(50, len(df)//150), max(80, len(df)//100)]:
            try:
                clusterer = hdbscan.HDBSCAN(min_cluster_size=mcs,
                                             min_samples=max(5, mcs//5),
                                             metric="euclidean",
                                             cluster_selection_method="eom",
                                             prediction_data=True)
                lbl = clusterer.fit_predict(cluster_input)
                n_cl = len(set(lbl)) - (1 if -1 in lbl else 0)
                if n_cl < 2: continue
                if (lbl == -1).any():
                    try:
                        soft = hdbscan.all_points_membership_vectors(clusterer)
                        for i in range(len(lbl)):
                            if lbl[i] == -1:
                                lbl[i] = int(np.argmax(soft[i])) if soft[i].sum() > 0 else 0
                    except Exception:
                        lbl[lbl == -1] = 0
                if len(set(lbl)) < 2: continue
                sil = silhouette_score(cluster_input, lbl, sample_size=min(3000, len(cluster_input)))
                print(f"    mcs={mcs:4d} -> k={len(set(lbl))}  sil={sil:.4f}")
                if sil > best_sil:
                    best_sil, best_labels = sil, lbl.copy()
            except Exception:
                pass
        if best_labels is None:
            from sklearn.cluster import KMeans
            best_labels = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=15).fit_predict(cluster_input)
            best_sil = np.nan
        df["cluster"] = best_labels
        print(f"  Clustering: k={len(set(best_labels))}  silhouette={best_sil:.4f}")
    else:
        from sklearn.cluster import KMeans
        df["cluster"] = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=15).fit_predict(cluster_input)

    kw_dict = _cluster_keywords(df, "cluster", "text_combined")
    return df, emb_2d, kw_dict


def _cluster_keywords(df, cluster_col, text_col, top_n=12):
    STOP = {"the", "a", "an", "of", "and", "in", "to", "for", "with", "on", "is", "are",
            "this", "that", "we", "our", "by", "as", "at", "be", "was", "were", "have",
            "has", "had", "from", "or", "not", "it", "its", "which", "can", "may", "also",
            "using", "used", "based", "study", "paper", "proposed", "method", "system",
            "model", "data", "result", "results", "analysis", "approach", "show", "effect"}
    kw_dict = {}
    for cid in sorted(df[cluster_col].unique()):
        texts = df[df[cluster_col] == cid][text_col].fillna("").tolist()
        tf = Counter()
        for t in texts:
            tf.update(w for w in re.findall(r"\b[a-z]{3,}\b", t.lower()) if w not in STOP)
        total = sum(tf.values()) or 1
        df_col = df[text_col].str.lower()
        scored = {}
        for w, cnt in tf.most_common(200):
            doc_freq = df_col.str.contains(r'\b' + w + r'\b', na=False).sum()
            scored[w] = (cnt / total) * math.log(len(df) / (doc_freq + 1))
        kw_dict[cid] = sorted(scored, key=scored.get, reverse=True)[:top_n]
    return kw_dict


# ==============================================================================
# Econometric helpers
# ==============================================================================
def _within_demean_fe(df, dv, controls, fe_cols):
    data = _clean_df(df, [dv] + controls + fe_cols)
    y_arr = data[dv].values.copy().astype(float)
    X_arr = data[controls].values.copy().astype(float)
    for _ in range(3):
        for fe in fe_cols:
            groups = data[fe].values
            for g in np.unique(groups):
                mask = groups == g
                y_arr[mask] -= y_arr[mask].mean()
                X_arr[mask] -= X_arr[mask].mean(axis=0)
    X_sm = sm.add_constant(pd.DataFrame(X_arr, columns=controls), has_constant="add")
    try:
        return OLS(pd.Series(y_arr, name=dv), X_sm).fit(cov_type="HC3")
    except Exception:
        return OLS(pd.Series(y_arr, name=dv), X_sm).fit()


def _iv2sls_primary(df, dv, endog, instrument, controls, label="IV"):
    data = _clean_df(df, [dv, endog, instrument] + controls)
    if len(data) < 50:
        return None, {}
    exog = sm.add_constant(data[controls])
    try:
        iv = IV2SLS(data[dv], exog, data[[endog]], data[[instrument]]).fit(cov_type="robust")
        try:
            fs_diag = iv.first_stage.diagnostics
            fs_f = float(fs_diag["f.stat"].values[0])
            fs_p = float(fs_diag["f.pval"].values[0])
            print(f"    [{label}] First-stage F={fs_f:.2f}  "
                  f"{'Strong' if fs_f > 10 else 'WEAK'}")
        except Exception:
            fs_f, fs_p = None, None
        return iv, {"fs_f": fs_f, "fs_p": fs_p}
    except Exception as e:
        print(f"    [{label}] IV failed: {e} -> OLS fallback")
        X = sm.add_constant(data[controls + [endog]])
        return OLS(data[dv], X).fit(cov_type="HC3"), {}


# ==============================================================================
# STEP 4 — Econometric analysis (main-text specifications; Q1/Table A5 excluded)
# ==============================================================================
def run_econometrics(df: pd.DataFrame):
    print("\n[STEP 4] Econometric Analysis")
    results = {}
    base_cols = ["log_patent", "log_citation", "oa_dummy", "Pub_Year", "post_policy",
                 "high_exposure", "ddd", "iv_shift_share", "pre_field_oa",
                 "recent", "log_field_div", "primary_field", "field_yr_trend"]
    clean = _clean_df(df, base_cols)

    le_pub = LabelEncoder()
    clean["pub_year_enc"] = le_pub.fit_transform(clean["Pub_Year"].astype(str))
    le_fld = LabelEncoder()
    clean["field_enc"] = le_fld.fit_transform(clean["primary_field"].astype(str))
    controls_base = ["log_citation", "oa_dummy", "recent", "log_field_div"]

    X1_data = _clean_df(clean, controls_base + ["log_patent"])
    X1 = sm.add_constant(X1_data[controls_base])
    results["m1_ols"] = OLS(X1_data["log_patent"], X1).fit(cov_type="HC3")

    results["m2_fe"] = _within_demean_fe(clean, "log_patent", controls_base,
                                          ["pub_year_enc", "field_enc"])

    m4, m4_diag = _iv2sls_primary(
        clean, "log_patent", "oa_dummy", "iv_shift_share",
        ["log_citation", "recent", "log_field_div"], "Shift-Share IV")
    results["m4_iv"] = m4
    results["m4_iv_diag"] = m4_diag

    m4b, m4b_diag = _iv2sls_primary(
        clean, "log_patent", "oa_dummy", "iv_residualized",
        ["log_citation", "recent", "log_field_div", "field_yr_trend"], "Residualized IV")
    results["m4b_iv_resid"] = m4b
    results["m4b_iv_resid_diag"] = m4b_diag

    X9_data = _clean_df(clean, controls_base + ["log_patent"])
    X9_df = sm.add_constant(X9_data[controls_base])
    qm = QuantReg(X9_data["log_patent"].values, X9_df.values)
    results["m9_quantile"] = {}
    for tau in [0.50, 0.75, 0.90, 0.95]:
        qr = qm.fit(q=tau, vcov="robust")
        qr._exog_names = list(X9_df.columns)
        results["m9_quantile"][tau] = qr

    print("  Econometrics complete")
    return results, clean


# ==============================================================================
# STEP 5 — Bartik IV validity suite
# ==============================================================================
def bartik_validity_suite(df: pd.DataFrame, results: dict) -> dict:
    print("\n[STEP 5] Bartik IV Validity Suite")
    validity = {}

    iv1_cols = ["log_patent", "iv_shift_share", "log_citation", "log_field_div",
                "field_yr_trend", "Pub_Year"]
    pre_data = _clean_df(df[df["Pub_Year"] < POLICY_YEAR], iv1_cols)
    if len(pre_data) > 100:
        X_pre = sm.add_constant(pre_data[["iv_shift_share", "log_citation",
                                           "log_field_div", "field_yr_trend"]])
        m_pre = OLS(pre_data["log_patent"], X_pre).fit(cov_type="HC3")
        b = _get_param(m_pre, "iv_shift_share")
        p = _get_pval(m_pre, "iv_shift_share")
        print(f"  Pre-trend: beta={b:+.4f}  p={p:.3f}  "
              f"{'PASS' if p > 0.1 else 'FAIL'}")
        validity["pre_trend"] = {"beta": float(b), "pval": float(p), "pass": bool(p > 0.1)}

    lofo_cols = ["log_patent", "oa_dummy", "iv_shift_share", "log_citation", "recent",
                 "log_field_div", "primary_field", "oa_rate_global_yr", "pre_field_oa", "Pub_Year"]
    lofo_clean = _clean_df(df, lofo_cols)
    lofo_coefs = []
    for fld in lofo_clean["primary_field"].unique()[:20]:
        sub = lofo_clean[lofo_clean["primary_field"] != fld].copy()
        pre_sub = sub[sub["Pub_Year"] < POLICY_YEAR].groupby("primary_field")["oa_dummy"].mean()
        sub["pre_field_oa_lofo"] = sub["primary_field"].map(pre_sub).fillna(sub["pre_field_oa"])
        sub["iv_lofo"] = sub["oa_rate_global_yr"].fillna(0) * sub["pre_field_oa_lofo"].fillna(0)
        sub_clean = _clean_df(sub, ["log_patent", "oa_dummy", "iv_lofo", "log_citation",
                                     "recent", "log_field_div"])
        if len(sub_clean) < 100: continue
        try:
            exog = sm.add_constant(sub_clean[["log_citation", "recent", "log_field_div"]])
            iv_lo = IV2SLS(sub_clean["log_patent"], exog, sub_clean[["oa_dummy"]],
                            sub_clean[["iv_lofo"]]).fit(cov_type="robust")
            lofo_coefs.append(float(iv_lo.params["oa_dummy"]))
        except Exception:
            pass
    if lofo_coefs:
        arr = np.array(lofo_coefs)
        print(f"  LOFO beta: mean={arr.mean():.4f}  std={arr.std():.4f}  "
              f"{'STABLE' if arr.std() < 0.05 else 'SENSITIVE'}")
        validity["lofo"] = {"mean": float(arr.mean()), "std": float(arr.std()),
                             "stable": bool(arr.std() < 0.05)}

    return validity


# ==============================================================================
# STEP 6 — Event study (Sun & Abraham IW)
# ==============================================================================
def sun_abraham_event_study(df: pd.DataFrame):
    print("\n[STEP 6] Sun & Abraham IW Event Study")
    es_cols = ["log_patent", "log_citation", "log_field_div", "Pub_Year", "cohort_year", "primary_field"]
    clean = _clean_df(df, es_cols)
    clean = clean[clean["cohort_year"] < 9999].copy()

    cohort_counts = clean["cohort_year"].value_counts()
    valid_cohorts = cohort_counts[cohort_counts >= 50].index.tolist()
    clean = clean[clean["cohort_year"].isin(valid_cohorts)].copy()
    clean["year_rel_int"] = (clean["Pub_Year"] - clean["cohort_year"]).clip(ES_WIN_PRE, ES_WIN_POST)
    clean_win = clean[clean["year_rel_int"].between(ES_WIN_PRE, ES_WIN_POST)].copy()

    ref_rel = -1
    all_iw_coefs, all_iw_ses = {}, {}
    for rel in sorted(clean_win["year_rel_int"].unique()):
        if rel == ref_rel: continue
        catt_list, se_list, n_list = [], [], []
        for g in valid_cohorts:
            sub = clean_win[(clean_win["cohort_year"] == g) &
                             (clean_win["year_rel_int"].isin([ref_rel, rel]))].copy()
            if len(sub) < 15: continue
            sub["is_rel"] = (sub["year_rel_int"] == rel).astype(int)
            sub_c = _clean_df(sub, ["log_patent", "is_rel", "log_citation", "log_field_div"])
            if len(sub_c) < 10: continue
            try:
                X = sm.add_constant(sub_c[["is_rel", "log_citation", "log_field_div"]])
                m = OLS(sub_c["log_patent"], X).fit(cov_type="HC3")
                coef = _get_param(m, "is_rel")
                se_val = float(m.bse["is_rel"]) if hasattr(m.bse, "__getitem__") else 0.02
                if np.isnan(coef) or np.isnan(se_val): continue
                catt_list.append(coef); se_list.append(max(se_val, 1e-6)); n_list.append(len(sub_c))
            except Exception:
                pass
        if catt_list:
            weights = np.array(n_list, dtype=float) / sum(n_list)
            all_iw_coefs[rel] = float(np.dot(weights, catt_list))
            all_iw_ses[rel] = float(np.sqrt(np.dot(weights**2, np.array(se_list)**2)))

    rel_vals = sorted(clean_win["year_rel_int"].unique())
    sa_rows = []
    for rel in rel_vals:
        if rel == ref_rel:
            sa_rows.append({"year_rel": rel, "coef": 0.0, "se": 0.0, "ci_lo": 0.0, "ci_hi": 0.0, "pval": 1.0})
        elif rel in all_iw_coefs:
            coef = all_iw_coefs[rel]; se = all_iw_ses[rel]
            pval = 2 * (1 - ss.norm.cdf(abs(coef / max(se, 1e-9))))
            sa_rows.append({"year_rel": rel, "coef": coef, "se": se,
                             "ci_lo": coef - 1.96*se, "ci_hi": coef + 1.96*se, "pval": pval})
    sa_df = pd.DataFrame(sa_rows).sort_values("year_rel")
    print(f"  IW event study complete: {len(sa_df)} periods")
    return sa_df


# ==============================================================================
# STEP 7 — Causal Forest HTE
# ==============================================================================
def causal_forest_hte(df: pd.DataFrame) -> dict:
    print("\n[STEP 7] Causal Forest HTE")
    X_features = ["log_citation", "log_field_div", "recent", "pre_field_oa",
                  "novelty_score", "interdisciplinarity", "disruptiveness",
                  "log_oa_x_citation", "pre_oa_x_recent", "citation_sq",
                  "patent_citation_ratio", "field_yr_trend"]
    X_features = [f for f in X_features if f in df.columns]
    clean = _clean_df(df, ["log_patent", "oa_dummy"] + X_features)
    Y = clean["log_patent"].values
    T = clean["oa_dummy"].values.astype(float)
    X = clean[X_features].fillna(0).values

    if HAS_ECONML:
        try:
            cf = CausalForestDML(n_estimators=300, min_samples_leaf=5, max_depth=6,
                                  random_state=SEED, n_jobs=-1, cv=5)
            cf.fit(Y, T, X=X, W=X[:, :4])
            te = cf.effect(X)
            return {"cate_mean": float(te.mean()), "cate_std": float(te.std()),
                    "te_vector": te, "method": "CausalForestDML"}
        except Exception as e:
            print(f"  CausalForestDML failed: {e} -> DR-Learner")

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)
    mu_hat = np.zeros_like(Y, dtype=float)
    ps_hat = np.zeros_like(Y, dtype=float)
    for train_idx, test_idx in kf.split(X_sc):
        ps_m = LogisticRegression(C=0.1, random_state=SEED, max_iter=500)
        ps_m.fit(X_sc[train_idx], T[train_idx])
        ps_hat[test_idx] = np.clip(ps_m.predict_proba(X_sc[test_idx])[:, 1], 0.05, 0.95)
        for t_val in [0, 1]:
            mask_t = T[train_idx] == t_val
            if mask_t.sum() < 20: continue
            gbm = GradientBoostingRegressor(n_estimators=150, max_depth=3,
                                             learning_rate=0.05, random_state=SEED)
            gbm.fit(X_sc[train_idx][mask_t], Y[train_idx][mask_t])
            test_mask = test_idx[T[test_idx] == t_val]
            if len(test_mask) > 0:
                mu_hat[test_mask] = gbm.predict(X_sc[test_mask])
    mu1 = np.where(T == 1, Y, mu_hat + (Y - mu_hat) / (ps_hat + 1e-9))
    mu0 = np.where(T == 0, Y, mu_hat - (Y - mu_hat) / (1 - ps_hat + 1e-9))
    pseudo = mu1 - mu0
    rf = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                random_state=SEED, n_jobs=-1)
    rf.fit(X_sc, pseudo)
    te = rf.predict(X_sc)
    print(f"  DR-Learner CATE: mean={te.mean():.4f}  std={te.std():.4f}")
    return {"cate_mean": float(te.mean()), "cate_std": float(te.std()),
            "te_vector": te, "method": "DR-Learner",
            "feature_importance": dict(zip(X_features, rf.feature_importances_.tolist()))}


# ==============================================================================
# STEP 8 — Descriptive statistics
# ==============================================================================
def descriptive_stats(df: pd.DataFrame):
    vars_desc = {
        "Citing_Patents": "Patent Citations",     "Citing_Works": "Academic Citations",
        "log_patent":     "log(Patent Cit.)",     "log_citation": "log(Academic Cit.)",
        "oa_dummy":       "Open Access",          "recent":       "Recent (>=2018)",
        "field_diversity": "Field Diversity",     "interdisciplinarity": "Interdiscipl.",
        "disruptiveness": "Disruptiveness",       "novelty_score": "Novelty",
        "top1pct":        "Superstar (Top 1%)",   "pre_field_oa": "Pre-Field OA Rate",
        "iv_shift_share": "IV (Shift-Share)",     "iv_residualized": "IV (Residualized)",
    }
    rows = []
    for var, label in vars_desc.items():
        if var not in df.columns: continue
        s = df[var].dropna()
        rows.append({"Variable": label, "N": f"{len(s):,}", "Mean": f"{s.mean():.3f}",
                     "SD": f"{s.std():.3f}", "Min": f"{s.min():.3f}",
                     "P50": f"{s.median():.3f}", "Max": f"{s.max():.3f}"})
    desc_df = pd.DataFrame(rows)
    desc_df.to_csv(OUT_DIR / "tables" / "table1_descriptive.csv", index=False, encoding="utf-8-sig")
    print("  table1_descriptive.csv written")
    return desc_df


# ==============================================================================
# Q3 — Sample window resolution (extraction vs. analysis vs. visualization)
# ==============================================================================
def resolve_sample_window(df_raw_analysis_sample: pd.DataFrame,
                           df_analysis_built: pd.DataFrame) -> dict:
    """
    df_raw_analysis_sample : CN + Citing_Patents>=1 sample, PRE build_indices
    df_analysis_built       : same sample, POST build_indices (has Pub_Year)
    """
    print("\n" + "=" * 65)
    print("[Q3] Sample Window Diagnostic")
    print("=" * 65)

    yr_extract  = df_raw_analysis_sample["Pub_Year"].value_counts().sort_index()
    yr_analysis = df_analysis_built["Pub_Year"].value_counts().sort_index()

    summary = pd.DataFrame({"extraction_n": yr_extract}).join(
        pd.DataFrame({"analysis_n": yr_analysis}), how="outer"
    ).fillna(0).astype(int).sort_index()

    stable_years = summary[summary["analysis_n"] >= 10].index
    reco_start = int(stable_years.min()) if len(stable_years) else None
    reco_end = int(summary.index.max())

    result = {
        "extraction_min_year": int(summary.index.min()),
        "extraction_max_year": reco_end,
        "analysis_n_before_1995": int(summary.loc[summary.index < 1995, "analysis_n"].sum()),
        "analysis_n_1995_1997": int(summary.loc[(summary.index >= 1995) &
                                                  (summary.index < 1998), "analysis_n"].sum()),
        "analysis_n_1998_plus": int(summary.loc[summary.index >= 1998, "analysis_n"].sum()),
        "recommended_visualization_start": reco_start,
        "recommended_visualization_end": reco_end,
    }

    print(summary.to_string())
    print("\n" + json.dumps(result, indent=2))

    summary.to_csv(OUT_DIR / "tables" / "table_sample_window_diagnostic.csv")
    with open(OUT_DIR / "tables" / "q3_sample_window_summary.json", "w") as f:
        json.dump(result, f, indent=2)
    print("  table_sample_window_diagnostic.csv written")
    print("  q3_sample_window_summary.json written")

    print(f"""
  -- SUGGESTED MANUSCRIPT TEXT (Sec 3.1.1) --------------------------------
  "Records were queried over {result['extraction_min_year']}-{result['extraction_max_year']}
   (extraction window). Of the {len(df_raw_analysis_sample):,} China-affiliated,
   patent-cited records retrieved, {result['analysis_n_before_1995']} predate 1995
   and {result['analysis_n_1995_1997']} fall in 1995-1997; because annual cell
   sizes below n=10 in these early years produce unstable point estimates,
   Figures 2 and 3 visualize the period from {reco_start} onward, while the
   regression sample retains the full {result['extraction_min_year']}-
   {result['extraction_max_year']} extraction window."
  --------------------------------------------------------------------------
  """)
    return result


# ==============================================================================
# Q4 — Gold / Green / Bronze OA colour breakdown (main analysis sample)
# ==============================================================================
def compute_oa_colour_breakdown(df_analysis_built: pd.DataFrame) -> dict:
    print("\n" + "=" * 65)
    print("[Q4] OA Colour Breakdown (main analysis sample)")
    print("=" * 65)

    if "OA_Colour" not in df_analysis_built.columns:
        print("  WARNING: 'OA_Colour' column not found -- skipping Q4.")
        return {}

    col = df_analysis_built["OA_Colour"].fillna("closed/unknown")
    counts = col.value_counts()
    pct = (counts / counts.sum() * 100).round(1)
    tbl = pd.DataFrame({"n": counts, "pct": pct})
    print(tbl)
    tbl.to_csv(OUT_DIR / "tables" / "table_oa_colour_breakdown.csv")
    print("  table_oa_colour_breakdown.csv written")

    oa_only = df_analysis_built[df_analysis_built["oa_dummy"] == 1]
    oa_only_colour = (oa_only["OA_Colour"].fillna("unknown")
                       .value_counts(normalize=True) * 100).round(1)
    print("\n  Among OA=TRUE papers only:")
    print(oa_only_colour)

    result = {"full_sample_pct": tbl.to_dict(), "oa_subsample_pct": oa_only_colour.to_dict()}
    with open(OUT_DIR / "tables" / "q4_oa_colour_breakdown.json", "w") as f:
        json.dump(result, f, indent=2, default=str)
    print("  q4_oa_colour_breakdown.json written")
    return result


# ==============================================================================
# STEP 9 — Selected figures
# ==============================================================================
def create_figures(df, results, sa_df, kw_dict, emb_2d, cf_results):
    print("\n[STEP 9] Figure Generation")

    try:
        corr_vars = ["log_patent", "log_citation", "oa_dummy", "recent",
                     "log_field_div", "top1pct", "disruptiveness",
                     "novelty_score", "interdisciplinarity", "iv_shift_share", "iv_residualized"]
        corr_vars = [v for v in corr_vars if v in df.columns]
        nice_names = [v.replace("log_", "log(").replace("_", " ").title() for v in corr_vars]
        corr_df = df[corr_vars].corr()
        fig, ax = plt.subplots(figsize=(13, 11))
        mask = np.triu(np.ones_like(corr_df, dtype=bool))
        cmap = LinearSegmentedColormap.from_list("rw", ["#C0392B", "white", "#1D4E89"])
        sns.heatmap(corr_df, mask=mask, annot=True, fmt=".2f", cmap=cmap,
                    center=0, ax=ax, linewidths=0.5, annot_kws={"size": 8},
                    xticklabels=nice_names, yticklabels=nice_names)
        ax.set_title("Correlation Matrix (Pearson r)", pad=14)
        ax.tick_params(axis="x", rotation=38, labelsize=8)
        plt.tight_layout()
        plt.savefig(OUT_DIR / "figures" / "01_descriptive" / "fig0_corr_matrix.png")
        plt.close()
        print("  fig0_corr_matrix.png written")
    except Exception as e:
        print(f"  WARNING: fig0 failed: {e}")

    try:
        yr = (df.groupby("Pub_Year")["Citing_Patents"]
                .agg(n="count", mean="mean", med="median")
                .reset_index().query("1995 <= Pub_Year <= 2024"))
        fig, ax1 = plt.subplots(figsize=(13, 5.5))
        ax1.bar(yr["Pub_Year"], yr["n"], color=C_BLUE, alpha=0.55, label="# Papers")
        ax1.set_ylabel("Number of Papers", color=C_BLUE)
        ax2 = ax1.twinx()
        ax2.plot(yr["Pub_Year"], yr["mean"], "o-", color=C_RED, lw=2.2, ms=5, label="Mean")
        ax2.plot(yr["Pub_Year"], yr["med"], "s--", color=C_ORANGE, lw=1.5, ms=4, label="Median")
        ax2.set_ylabel("Patent Citations", color=C_RED)
        ax1.axvline(POLICY_YEAR, ls="--", color=C_RED, lw=1.8, alpha=0.7)
        ax1.set_xlabel("Publication Year")
        ax1.set_title("Annual Trend: Patent-Cited Papers from Chinese Institutions")
        plt.tight_layout()
        plt.savefig(OUT_DIR / "figures" / "01_descriptive" / "fig1_annual_trend.png")
        plt.close()
        print("  fig1_annual_trend.png written")
    except Exception as e:
        print(f"  WARNING: fig1 failed: {e}")

    if sa_df is not None and len(sa_df) > 2:
        try:
            fig, ax = plt.subplots(figsize=(11, 5.5))
            pre = sa_df[sa_df["year_rel"] < 0]
            post = sa_df[sa_df["year_rel"] >= 0]
            ax.fill_between(pre["year_rel"], pre["ci_lo"], pre["ci_hi"], alpha=0.18, color=C_TEAL)
            ax.fill_between(post["year_rel"], post["ci_lo"], post["ci_hi"], alpha=0.18, color=C_BLUE)
            ax.plot(pre["year_rel"], pre["coef"], "o-", color=C_TEAL, lw=2.2, ms=7)
            ax.plot(post["year_rel"], post["coef"], "s-", color=C_BLUE, lw=2.2, ms=7)
            ax.axvline(-0.5, ls="--", color=C_RED, lw=2)
            ax.axhline(0, ls=":", color=C_GREY, lw=1.3)
            ax.set_xlabel("Year Relative to Policy (t=-1 reference)")
            ax.set_ylabel("Coefficient")
            ax.set_title("Sun & Abraham (2021) IW Event Study")
            plt.tight_layout()
            plt.savefig(OUT_DIR / "figures" / "02_main_results" / "fig4b_event_study.png")
            plt.close()
            print("  fig4b_event_study.png written")
        except Exception as e:
            print(f"  WARNING: fig4b failed: {e}")

    print("  (Additional figures generated by R pipeline -- see r/platform_analysis.R)")


# ==============================================================================
# STEP 10 — JSON summary
# ==============================================================================
def export_summary_json(results, df, cf_results, validity, q3_result, q4_result):
    def safe(v):
        if v is None or (isinstance(v, float) and math.isnan(v)): return None
        return float(v)

    summary = {
        "n_obs": int(df.shape[0]),
        "oa_rate": round(float(df["oa_dummy"].mean()), 4),
        "M1_OLS": {"beta_oa": safe(_get_param(results.get("m1_ols"), "oa_dummy")),
                   "pval_oa": safe(_get_pval(results.get("m1_ols"), "oa_dummy"))},
        "M4_IV": {"beta_oa": safe(_get_param(results.get("m4_iv"), "oa_dummy")),
                  "fs_F": safe((results.get("m4_iv_diag") or {}).get("fs_f"))},
        "M4b_IV": {"beta_oa": safe(_get_param(results.get("m4b_iv_resid"), "oa_dummy"))},
        "CATE": {"mean": safe(cf_results.get("cate_mean")),
                 "std": safe(cf_results.get("cate_std")),
                 "method": cf_results.get("method", "")},
        "IV_validity": validity,
        "Q3_sample_window": q3_result,
        "Q4_oa_colour": q4_result.get("full_sample_pct") if q4_result else None,
        "note": "Q1 (Table A5, zero-citation robustness) intentionally excluded "
                "from this pipeline; Q5 (C14 COVID cluster) intentionally not searched.",
    }
    with open(OUT_DIR / "tables" / "auto_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)
    print("  auto_summary.json written")
    return summary


# ==============================================================================
# MAIN
# ==============================================================================
def main(csv_path: Optional[str] = None):
    t0 = time.perf_counter()
    print("\n" + "*" * 65)
    print("  Open Access as Orthogonal Platform -- Unified Pipeline")
    print("  (Q1 excluded | Q3 + Q4 included | Q5 not searched)")
    print("*" * 65)

    path = csv_path or CSV_PATH

    # -- STEP 1: load raw (pre build_indices) analysis sample for Q3 -----------
    df_raw_analysis = load_data_raw_cn(path)

    # -- STEP 2: build_indices ---------------------------------------------------
    df = build_indices(df_raw_analysis.copy())

    # -- Q3: sample window (needs raw + built) -----------------------------------
    q3_result = resolve_sample_window(df_raw_analysis, df)

    # -- Q4: OA colour breakdown ---------------------------------------------------
    q4_result = compute_oa_colour_breakdown(df)

    # -- STEP 3: embedding + clustering (adds 'cluster' column) -------------------
    df, emb_2d, kw = embed_and_cluster(df)

    # -- STEP 8: descriptive stats --------------------------------------------------
    descriptive_stats(df)

    # -- STEP 4-7: econometrics / validity / event study / causal forest ----------
    results, clean = run_econometrics(df)
    validity = bartik_validity_suite(df, results)
    sa_df = sun_abraham_event_study(clean)
    cf_results = causal_forest_hte(df)

    # -- STEP 9-10: figures + summary -----------------------------------------------
    create_figures(df, results, sa_df, kw, emb_2d, cf_results)
    export_summary_json(results, df, cf_results, validity, q3_result, q4_result)

    # -- Save final analysis CSV -----------------------------------------------------
    # NOTE: this list is FIXED relative to the original script's `save_cols`,
    # which omitted OA_Colour / recent / log_field_div / pre_field_oa /
    # primary_field. Those columns are REQUIRED by
    # `oa_colour_heterogeneity_pipeline.py`'s REQUIRED_COLS, so omitting them
    # caused the downstream KeyError. They are included below.
    save_cols = [c for c in [
        "Institution", "Pub_Year", "Citing_Patents", "Citing_Works",
        "oa_dummy", "OA_Colour",                      # <- Q4 / colour pipeline
        "log_patent", "log_citation",
        "cluster", "top1pct",
        "disruptiveness", "novelty_score", "interdisciplinarity",
        "primary_field",                               # <- colour pipeline
        "recent",                                       # <- colour pipeline
        "log_field_div",                                # <- colour pipeline
        "high_exposure", "pre_field_oa",                # <- colour pipeline
        "iv_shift_share", "iv_residualized", "cohort_year",
        "field_yr_trend",
    ] if c in df.columns]

    missing_for_colour_pipeline = [c for c in [
        "log_patent", "log_citation", "oa_dummy", "OA_Colour", "Pub_Year",
        "recent", "log_field_div", "cluster", "pre_field_oa", "primary_field",
    ] if c not in save_cols]
    if missing_for_colour_pipeline:
        print(f"  WARNING: columns still missing for the colour heterogeneity "
              f"pipeline: {missing_for_colour_pipeline}")
    else:
        print("  ✓ All columns required by oa_colour_heterogeneity_pipeline.py's "
              "REQUIRED_COLS are present in analysis_final.csv")

    df[save_cols].to_csv(OUT_DIR / "tables" / "analysis_final.csv",
                          index=False, encoding="utf-8-sig")
    print(f"  analysis_final.csv written ({len(df):,} rows x {len(save_cols)} cols)")

    print(f"\nPipeline complete: {(time.perf_counter()-t0)/60:.1f} min  ->  {OUT_DIR}")
    print("\nNext step: run oa_colour_heterogeneity_pipeline.py with:")
    print(f"  python oa_colour_heterogeneity_pipeline.py --input {OUT_DIR}/tables/analysis_final.csv")

    return {
        "df": df, "results": results, "validity": validity,
        "sa_df": sa_df, "cf_results": cf_results,
        "q3_sample_window": q3_result, "q4_oa_colour": q4_result,
    }


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Open Access Orthogonal Platform — unified pipeline")
    parser.add_argument("--csv", type=str, default=None,
                         help="Path to the raw CSV (overrides CSV_PATH / env var)")
    args, _unknown = parser.parse_known_args()  # tolerate Jupyter's own -f kernel arg
    main(csv_path=args.csv)

  [PATCH] umap-learn/scikit-learn keyword compatibility patch applied (scoped to umap only)

*****************************************************************
  Open Access as Orthogonal Platform -- Unified Pipeline
  (Q1 excluded | Q3 + Q4 included | Q5 not searched)
*****************************************************************

[STEP 1] Data Loading + Preprocessing
  Encoding: [latin-1]
  [QC] Row Integrity Check
    Fully-empty rows dropped: 1,000
    Column-shifted rows flagged & dropped: 2
    Example shifted Lens IDs: ['019-882-760-622-986', '119-263-436-437-060']
    Recommendation: re-export these Lens IDs from Lens.org directly rather than attempting automated column-shift repair.
    Rows: 13,350 -> 12,348
  Total -> CN -> Patent>=1: 12,302 -> 12,302 -> 12,302
  Build complete: 12,302 rows x 75 cols
  OA rate: 57.1%  |  IV mean: 0.285  |  IV-Resid mean: 0.0000  std: 0.0256

[Q3] Sample Window Diagnostic
          extraction_n  analysis_n
Pub_Year                          

Batches: 100%|██████████| 193/193 [00:07<00:00, 25.83it/s]


  Embedding: (12302, 384)  (7.5s)
    mcs=  61 -> k=20  sil=0.2174
    mcs=  82 -> k=20  sil=0.2873
    mcs= 123 -> k=2  sil=-0.0592
  Clustering: k=20  silhouette=0.2873
  table1_descriptive.csv written

[STEP 4] Econometric Analysis
    [Shift-Share IV] First-stage F=1030.38  Strong
    [Residualized IV] First-stage F=43.26  Strong
  Econometrics complete

[STEP 5] Bartik IV Validity Suite
  Pre-trend: beta=+0.2710  p=0.000  FAIL
  LOFO beta: mean=0.1166  std=0.0015  STABLE

[STEP 6] Sun & Abraham IW Event Study
  IW event study complete: 6 periods

[STEP 7] Causal Forest HTE

[STEP 9] Figure Generation
  fig0_corr_matrix.png written
  fig1_annual_trend.png written
  fig4b_event_study.png written
  (Additional figures generated by R pipeline -- see r/platform_analysis.R)
  auto_summary.json written
  ✓ All columns required by oa_colour_heterogeneity_pipeline.py's REQUIRED_COLS are present in analysis_final.csv
  analysis_final.csv written (12,302 rows x 22 cols)

Pipeline complete: 6

In [1]:
#세부OA분석##

In [2]:
"""
OA-Colour Heterogeneity Pipeline — Open Access as Orthogonal Platform
========================================================================
Purpose
-------
Extends the main analysis by treating OA "colour" (gold / green / hybrid /
bronze / closed) not merely as a descriptive breakdown (Q4) but as a
THEORETICALLY MOTIVATED source of heterogeneity that lets the orthogonal
platform framework make an additional, falsifiable prediction the current
manuscript does not yet test.

Why this matters for the theory (not just a robustness add-on)
----------------------------------------------------------------
The manuscript's Section 3.1.4 argues OA is an orthogonal platform under
INSTITUTIONAL governance (funder mandates, publisher policy) rather than
market governance. But "OA" as coded (a single TRUE/FALSE dummy) actually
pools together colour categories with very different governance intensity:

  gold    - APC paid, publisher-hosted, immediate, typically MANDATE-DRIVEN
            (funder pays, institution tracks compliance) -> strongest
            institutional governance signal
  green   - self-archived by the author, often after an embargo, VOLUNTARY
            effort with no direct institutional payment -> weaker,
            author-discretionary governance signal
  hybrid  - subscription journal, individual article paid OA -> mixed
  bronze  - free-to-read but no clear license, publisher discretion, can
            revert to closed -> weakest/most fragile governance signal

If the orthogonal-platform mechanism (Section 2) is correct -- i.e., the
OA premium and its attenuation are driven by INSTITUTIONAL governance
intensity, not by accessibility alone -- then:

  P1: gold OA should show the largest premium (strongest, most
      unambiguous institutional signal to inventors/patent examiners)
  P2: gold OA should attenuate FASTEST with field-level penetration
      (it is the form most directly targeted by mandates, so it is also
      the form that becomes "modal" soonest within a maturing field --
      directly extends H2/H3 with a second, independent penetration
      channel)
  P3: bronze OA (weak/reversible governance signal, publisher
      discretion) should show the smallest and least stable premium --
      a natural placebo-like contrast that a pure "accessibility" account
      (the diffusion framework the paper argues against) would NOT predict,
      since bronze papers are just as freely readable as gold papers at
      the time of citation.

This pipeline tests P1-P3 directly, which converts the OA-colour
breakdown from a compliance/description item (Q4) into a genuine
STRENGTHENING of the paper's core theoretical contribution: it gives the
orthogonal-platform framework a prediction that the diffusion/visibility-
shock alternative cannot generate, addressing Reviewer 3's demand for
"predictions the prevailing account does not naturally accommodate."

Modules
-------
  C1. Compositional diagnostics: colour share over time & across clusters
      (also checks whether the "gold share" compositional shift itself
      partially explains platform maturation -- a mechanism check the
      current manuscript does not perform)
  C2. Covariate balance across colour categories (addresses quality-
      dilution alternative: are gold papers just higher quality to begin
      with, confounding any colour-specific premium?)
  C3. Multi-category OA-colour regression (frequentist): colour dummies
      vs. closed reference, plus colour x pre_field_oa interactions
      testing P2 directly
  C4. Bayesian hierarchical colour x cluster partial-pooling model
      (colour-specific posteriors with full uncertainty, extends the
      companion bayesian_extension_pipeline.py's B1 module)
  C5. Colour-specific attenuation curves (extends B2's spline approach,
      stratified by colour, testing P2/P3 with continuous penetration)
  C6. Superstar (top-1%) odds by colour (logistic) -- tests whether
      governance intensity predicts disproportionate upper-tail benefit,
      mirroring the existing superstar analysis in Section 4.1 but
      colour-resolved

Requirements
------------
    pip install numpy pandas scipy statsmodels matplotlib seaborn \
                 patsy pymc arviz --break-system-packages
    (pymc/arviz only required for C4; the rest is pure statsmodels/scipy
    and will run without them -- see HAS_BAYES guard below)

Usage
-----
    python oa_colour_heterogeneity_pipeline.py --input outputs/tables/analysis_final.csv

or from a notebook, reusing the already-built dataframe in memory:

    import oa_colour_heterogeneity_pipeline as ocp
    results = ocp.run_all(df)   # df must contain OA_Colour (see REQUIRED_COLS)

Changelog
---------
  * FIX (this version): `az.rhat(idata).max().to_array().max()` raised
    `AttributeError: 'DataTree' object has no attribute 'to_array'` on
    newer arviz/xarray versions, where `.max()` on the rhat Dataset can
    return a DataTree-like object instead of a plain xarray.Dataset with
    a `.to_array()` method. Replaced with a version-agnostic `_max_rhat()`
    helper that iterates `data_vars` directly instead of relying on the
    `to_array()`/`to_dataarray()` API, which differs across xarray
    releases. See `_max_rhat()` below and its two call sites in
    `bayesian_colour_hierarchy()`.
  * Also bumped `MAX_TREEDEPTH` and expose it in `pm.sample(...)` since
    the run reported all 4 chains hitting the default max tree depth
    (a soft warning, not an error, but worth relaxing given divergence-
    free sampling still needs deeper trees for this model).
  * FIX: `idata.to_netcdf(...)` raised
    `ValueError: cannot write NetCDF files ... because none of the
    suitable backend libraries (netCDF4, h5netcdf) are installed` when
    neither optional dependency is present. Sampling itself succeeds --
    only the cache write fails -- so added `_save_idata()`, which tries
    `to_netcdf()` first and falls back to a pickle file
    (`C4_idata.pkl`) if no netCDF backend is available, with an
    on-screen hint to `pip install netCDF4` (or `h5netcdf`) for standard
    `.nc` output next time.
"""

import os
import json
import warnings
import argparse
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import scipy.stats as ss

warnings.filterwarnings("ignore")

import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.multitest import multipletests

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import pymc as pm
    import arviz as az
    import pytensor.tensor as pt
    HAS_BAYES = True
except ImportError:
    HAS_BAYES = False
    print("  NOTE: pymc/arviz not found -> C4 (Bayesian colour x cluster model) "
          "will be skipped. Install with:\n"
          "  pip install pymc arviz --break-system-packages")

try:
    from patsy import dmatrix
    HAS_PATSY = True
except ImportError:
    HAS_PATSY = False
    print("  NOTE: patsy not found -> C5 (spline attenuation curves) will use a "
          "simpler polynomial basis instead of B-splines. "
          "pip install patsy --break-system-packages for the full version.")

# ==============================================================================
# CONFIG
# ==============================================================================
OUT = Path("outputs/colour_heterogeneity")
(OUT / "tables").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)

SEED = 2025
np.random.seed(SEED)
DRAWS, TUNE, CHAINS, TARGET_ACCEPT = 1500, 1500, 4, 0.9
MAX_TREEDEPTH = 12  # bumped from the pymc default of 10 -- the model reported
                     # all chains hitting max tree depth at the default value

REQUIRED_COLS = ["log_patent", "log_citation", "oa_dummy", "OA_Colour",
                  "Pub_Year", "recent", "log_field_div", "cluster",
                  "pre_field_oa", "primary_field"]

COLOUR_ORDER = ["closed/unknown", "bronze", "green", "hybrid", "gold"]
COLOUR_PALETTE = {
    "closed/unknown": "#7F8C8D", "bronze": "#B08D57", "green": "#1A7A4A",
    "hybrid": "#7D3C98", "gold": "#D4A017",
}
C_BLUE, C_RED, C_TEAL, C_GREY, C_ORANGE = (
    "#1D4E89", "#C0392B", "#148F77", "#7F8C8D", "#E67E22"
)
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
    "figure.dpi": 150, "savefig.dpi": 200, "savefig.bbox": "tight",
})


def _std(x):
    x = np.asarray(x, dtype=float)
    m, s = np.nanmean(x), np.nanstd(x)
    s = s if s > 1e-9 else 1.0
    return (x - m) / s, m, s


def _clean(df, cols):
    return (df.replace([np.inf, -np.inf], np.nan)
              .dropna(subset=cols).copy().reset_index(drop=True))


def _save_idata(idata, path: Path) -> None:
    """
    Save an ArviZ InferenceData object, tolerating environments where the
    optional netCDF backends aren't installed.

    `idata.to_netcdf(...)` requires either the `netCDF4` or `h5netcdf`
    package. When neither is present it raises:

        ValueError: cannot write NetCDF files with format='NETCDF4' because
        none of the suitable backend libraries (netCDF4, h5netcdf) are
        installed

    Rather than let this crash the whole C4 module (the sampling itself
    already succeeded at that point -- only the on-disk cache write fails),
    fall back to a plain pickle file so the posterior draws are still
    persisted and can be reloaded later.
    """
    try:
        idata.to_netcdf(str(path))
        print(f"  Saved InferenceData -> {path}")
    except (ValueError, ModuleNotFoundError) as e:
        pkl_path = path.with_suffix(".pkl")
        print(f"  NOTE: idata.to_netcdf() failed ({e}); no netCDF4/h5netcdf "
              f"backend is installed. Falling back to pickle: {pkl_path}\n"
              f"  For standard .nc output next time, run:\n"
              f"    pip install netCDF4 --break-system-packages\n"
              f"  (or: pip install h5netcdf --break-system-packages)")
        import pickle
        with open(pkl_path, "wb") as f:
            pickle.dump(idata, f)


def _max_rhat(idata) -> float:
    """
    Version-agnostic max-R-hat extraction.

    `az.rhat(idata)` returns an xarray.Dataset of per-variable R-hat arrays.
    Older arviz/xarray let you chain `.max().to_array().max()` to collapse
    that Dataset into a single scalar. On newer xarray releases, `.max()`
    on a Dataset can come back as a DataTree-like object that has no
    `.to_array()` (and no `.to_dataarray()` either, depending on version),
    which raised:

        AttributeError: 'DataTree' object has no attribute 'to_array'

    Iterating `.data_vars` directly and taking the max of each variable's
    values sidesteps the whole to_array()/to_dataarray() naming churn and
    works across arviz/xarray versions.
    """
    rhat_ds = az.rhat(idata)
    per_var_max = [float(np.asarray(da.values).max()) for da in rhat_ds.data_vars.values()]
    return max(per_var_max)


def _prep(df: pd.DataFrame) -> pd.DataFrame:
    """Normalizes OA_Colour categories and checks required columns exist."""
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise KeyError(
            f"Required columns missing: {missing}. This pipeline expects the "
            f"dataframe produced by oa_pipeline.build_indices() / "
            f"embed_and_cluster(), which includes 'OA_Colour'."
        )
    d = df.copy()
    d["OA_Colour"] = d["OA_Colour"].fillna("closed/unknown").astype(str).str.lower()
    d["OA_Colour"] = d["OA_Colour"].replace({"unknown": "closed/unknown"})
    valid = set(COLOUR_ORDER)
    unexpected = set(d["OA_Colour"].unique()) - valid
    if unexpected:
        print(f"  NOTE: collapsing unrecognized colour labels into "
              f"'closed/unknown': {unexpected}")
        d.loc[~d["OA_Colour"].isin(valid), "OA_Colour"] = "closed/unknown"
    d["OA_Colour"] = pd.Categorical(d["OA_Colour"], categories=COLOUR_ORDER, ordered=False)
    return d


# ==============================================================================
# C1 — Compositional diagnostics
# ==============================================================================
def compositional_diagnostics(df: pd.DataFrame) -> dict:
    """
    Two things this answers that the manuscript currently does not:

    (a) Does the GOLD share itself rise sharply post-2015, mechanically
        tracking the national OA mandate timeline (Section 3.1.2)? If so,
        this is direct compositional evidence for the "institutional
        governance" story (Section 3.1.4) -- gold is the colour category
        institutions can most directly compel and monitor, so its growth
        curve should track policy timing more tightly than green/bronze.

    (b) Which research clusters lean gold vs. green? Clusters with a
        heavy green-OA tilt (author-voluntary, discipline-norm-driven)
        vs. gold-tilt (mandate-compliance-driven) is a useful heterogeneity
        axis to cross-reference against the H3 maturity x knowledge-type
        matrix (Figure 9) in the main text.
    """
    print("\n[C1] Compositional Diagnostics: OA Colour Composition Over Time & Field")
    d = df.copy()

    # -- (a) Colour share by year --------------------------------------------
    yr_colour = (pd.crosstab(d["Pub_Year"], d["OA_Colour"], normalize="index") * 100)
    yr_colour = yr_colour.reindex(columns=COLOUR_ORDER, fill_value=0)
    yr_colour.to_csv(OUT / "tables" / "C1_colour_share_by_year.csv")

    # Growth-rate comparison: gold vs green share, pre- vs post-2015
    pre = yr_colour[yr_colour.index < 2015].mean()
    post = yr_colour[yr_colour.index >= 2015].mean()
    growth = pd.DataFrame({"pre_2015_mean_pct": pre, "post_2015_mean_pct": post,
                            "pp_change": post - pre})
    print(growth.round(2))
    growth.to_csv(OUT / "tables" / "C1_colour_growth_pre_post_policy.csv")

    fig, ax = plt.subplots(figsize=(12, 5.5))
    bottom = np.zeros(len(yr_colour))
    years = yr_colour.index.values
    for col in COLOUR_ORDER:
        ax.bar(years, yr_colour[col].values, bottom=bottom,
               color=COLOUR_PALETTE[col], label=col, width=0.85)
        bottom += yr_colour[col].values
    ax.axvline(2015, color="black", lw=1.6, ls="--", alpha=0.7, label="2015 policy")
    ax.set_ylabel("Share of papers (%)")
    ax.set_xlabel("Publication Year")
    ax.set_title("C1: OA Colour Composition by Year (stacked)")
    ax.legend(loc="upper left", ncol=3, fontsize=8.5)
    plt.tight_layout()
    plt.savefig(OUT / "figures" / "C1_colour_share_by_year.png")
    plt.close()

    # -- (b) Colour composition by top-N clusters ------------------------------
    top_clusters = d["cluster"].value_counts().head(15).index
    field_colour = (pd.crosstab(d.loc[d["cluster"].isin(top_clusters), "cluster"],
                                 d.loc[d["cluster"].isin(top_clusters), "OA_Colour"],
                                 normalize="index") * 100)
    field_colour = field_colour.reindex(columns=COLOUR_ORDER, fill_value=0)
    field_colour = field_colour.reindex(
        field_colour["gold"].sort_values(ascending=False).index)
    field_colour.to_csv(OUT / "tables" / "C1_colour_share_by_cluster.csv")

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(field_colour, annot=True, fmt=".0f", cmap="YlOrBr",
                cbar_kws={"label": "% of cluster papers"}, ax=ax,
                linewidths=0.4)
    ax.set_title("C1: OA Colour Composition by Research Cluster "
                  "(top 15 clusters by N, sorted by gold share)")
    ax.set_xlabel("OA Colour"); ax.set_ylabel("Cluster")
    plt.tight_layout()
    plt.savefig(OUT / "figures" / "C1_colour_by_cluster_heatmap.png")
    plt.close()

    result = {
        "growth_table": growth.round(3).to_dict(),
        "gold_pp_change_2015": float(growth.loc["gold", "pp_change"]),
        "green_pp_change_2015": float(growth.loc["green", "pp_change"]),
    }
    print(f"  Gold share change (pre->post 2015): {result['gold_pp_change_2015']:+.1f} pp")
    print(f"  Green share change (pre->post 2015): {result['green_pp_change_2015']:+.1f} pp")
    with open(OUT / "tables" / "C1_compositional_summary.json", "w") as f:
        json.dump(result, f, indent=2)
    print("  C1_colour_share_by_year.png, C1_colour_by_cluster_heatmap.png, "
          "C1_compositional_summary.json written")
    return result


# ==============================================================================
# C2 — Covariate balance across colour categories (quality-dilution check)
# ==============================================================================
def covariate_balance_by_colour(df: pd.DataFrame) -> dict:
    """
    Directly addresses the quality-dilution alternative explanation
    (Section 3.5.2 in the main text) at the colour level: if gold OA papers
    are simply higher-quality/more-cited papers from well-funded labs
    (APC affordability confound), the colour-specific premium in C3/C4
    would be an artifact of selection rather than governance intensity.
    This table lets readers see directly whether/how much colours differ
    on observable quality proxies BEFORE looking at the outcome regression.
    """
    print("\n[C2] Covariate Balance Across OA Colour Categories")
    quality_vars = ["log_citation", "novelty_score", "disruptiveness",
                     "interdisciplinarity", "log_field_div", "top1pct"]
    quality_vars = [v for v in quality_vars if v in df.columns]
    d = _clean(df, quality_vars + ["OA_Colour"])

    rows = []
    for v in quality_vars:
        grp_means = d.groupby("OA_Colour", observed=True)[v].mean()
        grp_sds = d.groupby("OA_Colour", observed=True)[v].std()
        f_stat, p_val = ss.f_oneway(*[d.loc[d["OA_Colour"] == c, v].values
                                       for c in COLOUR_ORDER
                                       if (d["OA_Colour"] == c).sum() > 5])
        row = {"Variable": v, "ANOVA_F": round(f_stat, 3), "ANOVA_p": round(p_val, 5)}
        for c in COLOUR_ORDER:
            row[f"{c}_mean"] = round(grp_means.get(c, np.nan), 3)
        rows.append(row)
    balance_tbl = pd.DataFrame(rows)
    print(balance_tbl.to_string(index=False))
    balance_tbl.to_csv(OUT / "tables" / "C2_covariate_balance.csv", index=False)

    # Pairwise gold-vs-green comparison (the theoretically key contrast, P1/P3)
    pairwise_rows = []
    for v in quality_vars:
        for c1, c2 in combinations(["gold", "green", "hybrid", "bronze"], 2):
            x1 = d.loc[d["OA_Colour"] == c1, v].dropna()
            x2 = d.loc[d["OA_Colour"] == c2, v].dropna()
            if len(x1) < 10 or len(x2) < 10:
                continue
            t, p = ss.ttest_ind(x1, x2, equal_var=False)
            pairwise_rows.append({"Variable": v, "Colour_1": c1, "Colour_2": c2,
                                   "mean_diff": round(x1.mean() - x2.mean(), 3),
                                   "t_stat": round(t, 3), "p_raw": p})
    pw = pd.DataFrame(pairwise_rows)
    if len(pw) > 0:
        pw["p_fdr"] = multipletests(pw["p_raw"], method="fdr_bh")[1]
        pw["p_raw"] = pw["p_raw"].round(5)
        pw["p_fdr"] = pw["p_fdr"].round(5)
    pw.to_csv(OUT / "tables" / "C2_pairwise_colour_comparisons.csv", index=False)
    print(f"\n  Pairwise comparisons (FDR-corrected) written: "
          f"C2_pairwise_colour_comparisons.csv")

    result = {"anova_summary": balance_tbl.to_dict(orient="records"),
              "n_significant_pairwise_fdr05": int((pw["p_fdr"] < 0.05).sum()) if len(pw) else 0}
    with open(OUT / "tables" / "C2_balance_summary.json", "w") as f:
        json.dump(result, f, indent=2, default=str)
    print("  C2_covariate_balance.csv, C2_balance_summary.json written")
    return result


# ==============================================================================
# C3 — Multi-category OA-colour regression (frequentist)
# ==============================================================================
def colour_regression(df: pd.DataFrame) -> dict:
    """
    Tests P1 (gold > green > hybrid > bronze in premium magnitude) and P2
    (colour x penetration interaction: does the fastest-attenuating colour
    correspond to the colour with the strongest institutional-mandate
    signal, i.e. gold?) in a single regression framework with closed/
    unknown as the reference category.

    Model:
        log_patent ~ C(OA_Colour) + C(OA_Colour):pre_field_oa
                     + log_citation + recent + log_field_div
                     + field & year fixed effects (via cluster + year dummies)
    """
    print("\n[C3] Multi-Category OA-Colour Regression")
    d = _clean(df, ["log_patent", "log_citation", "recent", "log_field_div",
                     "pre_field_oa", "OA_Colour", "cluster", "Pub_Year"])
    print(f"  N={len(d):,}")

    colour_dum = pd.get_dummies(d["OA_Colour"], prefix="colour", drop_first=False)
    colour_dum = colour_dum.drop(columns=["colour_closed/unknown"])  # reference category
    colour_int = colour_dum.multiply(d["pre_field_oa"].values, axis=0)
    colour_int.columns = [c.replace("colour_", "colour_x_penetration_") for c in colour_int.columns]

    log_cit, *_ = _std(d["log_citation"].values)
    log_fdiv, *_ = _std(d["log_field_div"].values)

    X = pd.concat([
        colour_dum.astype(float).reset_index(drop=True),
        colour_int.astype(float).reset_index(drop=True),
        pd.Series(log_cit, name="log_citation_z"),
        pd.Series(d["recent"].values.astype(float), name="recent"),
        pd.Series(log_fdiv, name="log_field_div_z"),
        pd.Series(d["pre_field_oa"].values, name="pre_field_oa"),
    ], axis=1)
    X = sm.add_constant(X)
    y = d["log_patent"].values

    model = OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": d["cluster"].values})
    print(model.summary().tables[1])

    coef_rows = []
    for colour in ["gold", "green", "hybrid", "bronze"]:
        main = f"colour_{colour}"
        inter = f"colour_x_penetration_{colour}"
        if main not in model.params.index:
            continue
        coef_rows.append({
            "Colour": colour,
            "Level_effect": round(model.params[main], 4),
            "Level_p": round(model.pvalues[main], 4),
            "Penetration_interaction": round(model.params.get(inter, np.nan), 4),
            "Penetration_interaction_p": round(model.pvalues.get(inter, np.nan), 4),
        })
    coef_tbl = pd.DataFrame(coef_rows).sort_values("Level_effect", ascending=False)
    print("\n" + coef_tbl.to_string(index=False))
    coef_tbl.to_csv(OUT / "tables" / "C3_colour_regression_coefs.csv", index=False)

    # P1 test: is gold's level effect significantly larger than green's?
    p1_supported = None
    if "gold" in coef_tbl["Colour"].values and "green" in coef_tbl["Colour"].values:
        g_gold = coef_tbl.loc[coef_tbl["Colour"] == "gold", "Level_effect"].iloc[0]
        g_green = coef_tbl.loc[coef_tbl["Colour"] == "green", "Level_effect"].iloc[0]
        p1_supported = bool(g_gold > g_green)
    # P2 test: is gold's penetration interaction the most negative (fastest attenuation)?
    p2_supported = None
    if len(coef_tbl.dropna(subset=["Penetration_interaction"])) > 1:
        most_negative_colour = coef_tbl.loc[
            coef_tbl["Penetration_interaction"].idxmin(), "Colour"]
        p2_supported = bool(most_negative_colour == "gold")

    result = {
        "coefficients": coef_tbl.to_dict(orient="records"),
        "P1_gold_gt_green_level_effect": p1_supported,
        "P2_gold_fastest_attenuation": p2_supported,
        "n_obs": int(len(d)),
    }
    print(f"\n  P1 (gold level-effect > green level-effect): {p1_supported}")
    print(f"  P2 (gold shows fastest/most-negative penetration interaction): {p2_supported}")

    with open(OUT / "tables" / "C3_colour_regression_summary.json", "w") as f:
        json.dump(result, f, indent=2, default=str)

    # Forest plot of colour level effects
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ci = model.conf_int()
    for i, row in coef_tbl.iterrows():
        colour = row["Colour"]
        main = f"colour_{colour}"
        lo, hi = ci.loc[main, 0], ci.loc[main, 1]
        ax.errorbar(row["Level_effect"], colour, xerr=[[row["Level_effect"] - lo],
                                                         [hi - row["Level_effect"]]],
                    fmt="o", color=COLOUR_PALETTE.get(colour, C_GREY), ms=10, capsize=4)
    ax.axvline(0, color="black", lw=1)
    ax.set_xlabel("OA-colour level effect on log(1+patent citations) "
                  "(ref. = closed/unknown)")
    ax.set_title("C3: Colour-Specific OA Level Effects (95% CI, field-clustered SE)")
    plt.tight_layout()
    plt.savefig(OUT / "figures" / "C3_colour_forest_plot.png")
    plt.close()

    print("  C3_colour_regression_coefs.csv, C3_colour_forest_plot.png, "
          "C3_colour_regression_summary.json written")
    return result


# ==============================================================================
# C4 — Bayesian hierarchical colour x cluster partial-pooling model
# ==============================================================================
def bayesian_colour_hierarchy(df: pd.DataFrame) -> dict:
    """
    Extends the companion bayesian_extension_pipeline.py's B1 module:
    instead of one OA dummy, this fits a full posterior for EACH colour
    category simultaneously, with cluster-level partial pooling on top,
    giving posterior probabilities like P(gold effect > green effect)
    directly -- the correct way to compare P1/P3 with full uncertainty
    rather than just comparing point estimates in C3.
    """
    if not HAS_BAYES:
        print("\n[C4] SKIPPED (pymc/arviz not installed)")
        return {}
    print("\n[C4] Bayesian Hierarchical Colour x Cluster Partial Pooling")
    d = _clean(df, ["log_patent", "log_citation", "recent", "log_field_div",
                     "OA_Colour", "cluster"])
    clusters, cluster_idx = np.unique(d["cluster"].values, return_inverse=True)
    colours, colour_idx = np.unique(d["OA_Colour"].astype(str).values, return_inverse=True)
    n_clusters, n_colours = len(clusters), len(colours)
    print(f"  N={len(d):,}  clusters={n_clusters}  colours={list(colours)}")

    y = d["log_patent"].values
    log_cit, *_ = _std(d["log_citation"].values)
    recent = d["recent"].values.astype(float)
    log_fdiv, *_ = _std(d["log_field_div"].values)

    with pm.Model() as model:
        # Colour effects: hierarchically pooled ACROSS colours (shrinks rare
        # categories like bronze toward the grand mean) AND across clusters
        # within colour (partial pooling on the field dimension too).
        mu_colour = pm.Normal("mu_colour", 0, 1)
        tau_colour = pm.HalfNormal("tau_colour", 0.5)
        colour_effect = pm.Normal("colour_effect", mu_colour, tau_colour, shape=n_colours)

        tau_cluster = pm.HalfNormal("tau_cluster", 0.5)
        cluster_offset = pm.Normal("cluster_offset", 0, tau_cluster, shape=n_clusters)

        alpha0 = pm.Normal("alpha0", 0, 2)
        gamma = pm.Normal("gamma_log_cit", 0, 1)
        delta = pm.Normal("delta_recent", 0, 1)
        theta = pm.Normal("theta_log_fdiv", 0, 1)
        sigma = pm.HalfNormal("sigma", 1)

        mu = (alpha0 + colour_effect[colour_idx] + cluster_offset[cluster_idx]
              + gamma * log_cit + delta * recent + theta * log_fdiv)
        pm.Normal("y_obs", mu, sigma, observed=y)

        idata = pm.sample(draws=DRAWS, tune=TUNE, chains=CHAINS,
                           target_accept=TARGET_ACCEPT, max_treedepth=MAX_TREEDEPTH,
                           random_seed=SEED)

    # --- FIX: version-agnostic max R-hat extraction (see _max_rhat() docstring) ---
    rhat_max = _max_rhat(idata)
    print(f"  Convergence: max R-hat={rhat_max:.4f}")
    colour_post = idata.posterior["colour_effect"].values.reshape(-1, n_colours)

    summary_rows = []
    for i, c in enumerate(colours):
        post = colour_post[:, i]
        summary_rows.append({
            "Colour": c, "posterior_mean": float(post.mean()),
            "hdi_2.5": float(np.percentile(post, 2.5)),
            "hdi_97.5": float(np.percentile(post, 97.5)),
        })
    summ_df = pd.DataFrame(summary_rows).sort_values("posterior_mean", ascending=False)
    print(summ_df.to_string(index=False))
    summ_df.to_csv(OUT / "tables" / "C4_colour_posterior_summary.csv", index=False)

    # Pairwise posterior probability comparisons (the key P1/P3 test)
    idx = {c: i for i, c in enumerate(colours)}
    pairwise = {}
    for c1, c2 in combinations([c for c in ["gold", "green", "hybrid", "bronze"]
                                  if c in idx], 2):
        diff = colour_post[:, idx[c1]] - colour_post[:, idx[c2]]
        pairwise[f"P({c1} > {c2})"] = float((diff > 0).mean())
    print("\n  Pairwise posterior probabilities:")
    for k, v in pairwise.items():
        print(f"    {k} = {v:.4f}")

    result = {
        "colour_posterior_summary": summ_df.to_dict(orient="records"),
        "pairwise_probabilities": pairwise,
        "rhat_max": rhat_max,
    }
    with open(OUT / "tables" / "C4_bayesian_colour_hierarchy.json", "w") as f:
        json.dump(result, f, indent=2)

    fig, ax = plt.subplots(figsize=(8.5, 5))
    for i, c in enumerate(colours):
        sns.kdeplot(colour_post[:, i], ax=ax,
                    color=COLOUR_PALETTE.get(c, C_GREY), lw=2.2, label=c, fill=True, alpha=0.15)
    ax.axvline(0, color="black", lw=1, ls=":")
    ax.set_xlabel("Colour effect on log(1+patent citations), posterior")
    ax.set_title("C4: Bayesian Posterior Distributions by OA Colour "
                  "(cluster-partial-pooled)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT / "figures" / "C4_colour_posteriors.png")
    plt.close()

    _save_idata(idata, OUT / "tables" / "C4_idata.nc")
    print("  C4_colour_posterior_summary.csv, C4_colour_posteriors.png, "
          "C4_bayesian_colour_hierarchy.json written")
    return result


# ==============================================================================
# C5 — Colour-specific attenuation curves
# ==============================================================================
def colour_attenuation_curves(df: pd.DataFrame, n_knots=6) -> dict:
    """
    Extends the companion pipeline's B2 spline attenuation model,
    stratified by colour: is gold's premium-vs-penetration curve steeper
    (attenuates faster) than green's? This is the continuous-outcome
    version of P2, complementing C3's linear interaction test with a
    fully flexible functional form. Colours with too few observations for
    a stable spline (bronze, hybrid) are pooled into an "other OA" category
    for this module only, to avoid overfitting sparse cells.
    """
    print("\n[C5] Colour-Specific Attenuation Curves")
    d = _clean(df, ["log_patent", "log_citation", "recent", "log_field_div",
                     "pre_field_oa", "OA_Colour"])
    d["colour_group"] = d["OA_Colour"].astype(str).replace(
        {"bronze": "other_oa", "hybrid": "other_oa"})
    groups = [g for g in ["gold", "green", "other_oa"] if (d["colour_group"] == g).sum() > 200]
    print(f"  Fitting curves for: {groups} "
          f"(N per group: {d['colour_group'].value_counts().to_dict()})")

    log_cit_all, *_ = _std(d["log_citation"].values)
    log_fdiv_all, *_ = _std(d["log_field_div"].values)
    d["log_cit_z"] = log_cit_all
    d["log_fdiv_z"] = log_fdiv_all

    x_grid = np.linspace(d["pre_field_oa"].min(), d["pre_field_oa"].max(), 150)
    curves = {}

    fig, ax = plt.subplots(figsize=(10, 6))
    for g in groups:
        sub = d[d["colour_group"] == g].copy()
        x = sub["pre_field_oa"].values
        y = sub["log_patent"].values

        if HAS_PATSY and len(sub) > 300:
            knots = np.quantile(x, np.linspace(0.1, 0.9, n_knots))
            basis = np.asarray(dmatrix(
                "bs(x, knots=knots, degree=3, include_intercept=True) - 1",
                {"x": x, "knots": knots}))
            basis_grid = np.asarray(dmatrix(
                "bs(x, knots=knots, degree=3, include_intercept=True) - 1",
                {"x": x_grid, "knots": knots}))
        else:
            # Fallback: cubic polynomial basis (adequate for a smooth
            # monotone-ish attenuation curve without patsy)
            basis = np.vstack([x, x**2, x**3]).T
            basis_grid = np.vstack([x_grid, x_grid**2, x_grid**3]).T
            basis = sm.add_constant(basis)
            basis_grid = sm.add_constant(basis_grid)

        controls = sub[["log_cit_z", "recent", "log_fdiv_z"]].values.astype(float)
        X = np.hstack([basis, controls])
        model = OLS(y, X).fit(cov_type="HC3")

        # predicted premium along the grid, controls held at sample means
        ctrl_means = controls.mean(axis=0)
        X_grid = np.hstack([basis_grid, np.tile(ctrl_means, (len(x_grid), 1))])
        pred = model.predict(X_grid)
        pred_se = np.sqrt(np.einsum("ij,jk,ik->i", X_grid, model.cov_params(), X_grid))
        lo, hi = pred - 1.96 * pred_se, pred + 1.96 * pred_se

        ax.plot(x_grid, pred, color=COLOUR_PALETTE.get(g, C_GREY), lw=2.4, label=g)
        ax.fill_between(x_grid, lo, hi, color=COLOUR_PALETTE.get(g, C_GREY), alpha=0.15)

        slope_start_end = float(pred[-1] - pred[0])
        curves[g] = {
            "premium_at_min_penetration": float(pred[0]),
            "premium_at_max_penetration": float(pred[-1]),
            "total_decline": slope_start_end,
            "n": int(len(sub)),
        }

    ax.axhline(0, color="black", lw=1, ls=":")
    ax.set_xlabel("Field-level pre-2013 OA penetration (pre_field_oa)")
    ax.set_ylabel("Predicted log(1+patent citations), colour-specific curve")
    ax.set_title("C5: Colour-Specific Premium vs. Field-Level Platform Penetration")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT / "figures" / "C5_colour_attenuation_curves.png")
    plt.close()

    print(pd.DataFrame(curves).T.to_string())
    with open(OUT / "tables" / "C5_colour_attenuation_curves.json", "w") as f:
        json.dump(curves, f, indent=2)
    print("  C5_colour_attenuation_curves.png, C5_colour_attenuation_curves.json written")

    # P2 check: does gold decline the most (most negative total_decline)?
    if "gold" in curves and len(curves) > 1:
        steepest = min(curves, key=lambda k: curves[k]["total_decline"])
        print(f"\n  Steepest decline: {steepest}  "
              f"(P2 -- 'gold attenuates fastest' -- "
              f"{'SUPPORTED' if steepest == 'gold' else 'NOT supported'})")
    return curves


# ==============================================================================
# C6 — Superstar (top-1%) odds by colour
# ==============================================================================
def superstar_odds_by_colour(df: pd.DataFrame) -> dict:
    """
    Mirrors the main text's superstar analysis (OR=1.84 for OA overall)
    but resolved by colour. If governance intensity drives cross-side
    amplification for high-visibility papers (Section 4.1's interaction
    finding), gold should show the largest superstar odds ratio.
    """
    print("\n[C6] Superstar (Top-1%) Odds by OA Colour")
    if "top1pct" not in df.columns:
        print("  'top1pct' column not found -- skipping C6.")
        return {}
    d = _clean(df, ["top1pct", "OA_Colour", "log_citation", "log_field_div"])

    import statsmodels.api as sm2
    colour_dum = pd.get_dummies(d["OA_Colour"], prefix="colour", drop_first=False)
    colour_dum = colour_dum.drop(columns=["colour_closed/unknown"])
    log_cit, *_ = _std(d["log_citation"].values)
    log_fdiv, *_ = _std(d["log_field_div"].values)

    X = pd.concat([colour_dum.astype(float).reset_index(drop=True),
                   pd.Series(log_cit, name="log_citation_z"),
                   pd.Series(log_fdiv, name="log_field_div_z")], axis=1)
    X = sm2.add_constant(X)
    y = d["top1pct"].values

    logit = sm2.Logit(y, X).fit(disp=0)
    print(logit.summary2().tables[1])

    or_tbl = pd.DataFrame({
        "Colour": [c.replace("colour_", "") for c in colour_dum.columns],
        "Odds_Ratio": np.exp(logit.params[colour_dum.columns].values),
        "CI_lo": np.exp(logit.conf_int().loc[colour_dum.columns, 0].values),
        "CI_hi": np.exp(logit.conf_int().loc[colour_dum.columns, 1].values),
        "p_value": logit.pvalues[colour_dum.columns].values,
    }).sort_values("Odds_Ratio", ascending=False)
    print(or_tbl.to_string(index=False))
    or_tbl.to_csv(OUT / "tables" / "C6_superstar_odds_by_colour.csv", index=False)

    fig, ax = plt.subplots(figsize=(7.5, 4))
    for _, row in or_tbl.iterrows():
        ax.errorbar(row["Odds_Ratio"], row["Colour"],
                    xerr=[[row["Odds_Ratio"] - row["CI_lo"]],
                          [row["CI_hi"] - row["Odds_Ratio"]]],
                    fmt="o", color=COLOUR_PALETTE.get(row["Colour"], C_GREY),
                    ms=10, capsize=4)
    ax.axvline(1, color="black", lw=1)
    ax.set_xlabel("Odds ratio: P(top-1% patent-cited), ref. = closed/unknown")
    ax.set_title("C6: Superstar Odds by OA Colour")
    plt.tight_layout()
    plt.savefig(OUT / "figures" / "C6_superstar_odds_forest.png")
    plt.close()

    result = {"odds_ratios": or_tbl.to_dict(orient="records")}
    with open(OUT / "tables" / "C6_superstar_odds.json", "w") as f:
        json.dump(result, f, indent=2)
    print("  C6_superstar_odds_by_colour.csv, C6_superstar_odds_forest.png written")
    return result


# ==============================================================================
# ORCHESTRATION
# ==============================================================================
def run_all(df: pd.DataFrame) -> dict:
    print("\n" + "*" * 70)
    print("  OA-Colour Heterogeneity Pipeline")
    print("  Testing: P1 (gold > green level effect), P2 (gold attenuates")
    print("  fastest), P3 (bronze weakest/least stable) -- see module docstrings")
    print("*" * 70)

    d = _prep(df)
    results = {}
    results["C1_composition"] = compositional_diagnostics(d)
    results["C2_balance"] = covariate_balance_by_colour(d)
    results["C3_regression"] = colour_regression(d)
    results["C4_bayesian"] = bayesian_colour_hierarchy(d)
    results["C5_attenuation"] = colour_attenuation_curves(d)
    results["C6_superstar"] = superstar_odds_by_colour(d)

    with open(OUT / "tables" / "colour_heterogeneity_summary_all.json", "w") as f:
        json.dump(results, f, indent=2, default=str)

    print("\n" + "=" * 70)
    print("  ALL COLOUR-HETEROGENEITY MODULES COMPLETE -- see "
          f"{OUT}/")
    print("=" * 70)

    print("""
  -- SUGGESTED MANUSCRIPT INTEGRATION -----------------------------------
  Add a new subsection (suggested: Section 4.6, "Governance-Intensity
  Heterogeneity by OA Colour") presenting C3's forest plot and C5's
  attenuation curves as a SECOND, independent test of the orthogonal-
  platform mechanism -- distinct from the H3 knowledge-type heterogeneity
  matrix (Figure 9), since it varies governance intensity directly rather
  than inferring it from field maturity alone. This is a genuinely novel
  contribution (not in Reviewer 1/2/3's original comments) that
  preemptively answers Reviewer 3's demand for predictions the diffusion
  framework cannot generate: pure accessibility (diffusion) predicts NO
  difference between gold and bronze, since both are equally readable at
  citation time: only the institutional-governance account predicts the
  gold > green > hybrid > bronze ordering tested here.
  ------------------------------------------------------------------------
    """)
    return results


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="OA-colour heterogeneity pipeline")
    parser.add_argument("--input", type=str,
                         default="outputs/tables/analysis_final.csv",
                         help="Path to the built/clustered analysis dataframe "
                              "(must contain OA_Colour; from oa_pipeline.py's "
                              "load_data()/build_indices())")
    parser.add_argument("--fast", action="store_true",
                         help="Short MCMC chains for C4 pilot run")
    args, _ = parser.parse_known_args()

    if args.fast:
        DRAWS, TUNE, CHAINS = 600, 600, 2
        print("  [FAST MODE] short chains for C4 -- re-run without --fast "
              "before using numbers in the manuscript.")

    if not os.path.exists(args.input):
        raise FileNotFoundError(
            f"{args.input} not found. Run oa_pipeline.py first, or note that "
            f"'OA_Colour' must be present -- it is created in load_data(), so "
            f"make sure you saved a version of the dataframe that includes it "
            f"(the default analysis_final.csv save_cols list in oa_pipeline.py "
            f"does NOT currently include OA_Colour -- add it there or pass "
            f"--input pointing at df_clustered_for_appendix.csv from "
            f"reviewer_response_pipeline.py, which does retain it)."
        )
    df_in = pd.read_csv(args.input)
    run_all(df_in)


**********************************************************************
  OA-Colour Heterogeneity Pipeline
  Testing: P1 (gold > green level effect), P2 (gold attenuates
  fastest), P3 (bronze weakest/least stable) -- see module docstrings
**********************************************************************

[C1] Compositional Diagnostics: OA Colour Composition Over Time & Field
                pre_2015_mean_pct  post_2015_mean_pct  pp_change
OA_Colour                                                       
closed/unknown              64.97               43.86     -21.11
bronze                       6.16                2.41      -3.75
green                       14.05                7.59      -6.47
hybrid                       1.51                3.16       1.65
gold                        13.30               42.98      29.68
  Gold share change (pre->post 2015): +29.7 pp
  Green share change (pre->post 2015): -6.5 pp
  C1_colour_share_by_year.png, C1_colour_by_cluster_heatmap.png, C1

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu_colour, tau_colour, colour_effect, tau_cluster, cluster_offset, alpha0, gamma_log_cit, delta_recent, theta_log_fdiv, sigma]


Sampling 4 chains for 1_500 tune and 1_500 draw iterations (6_000 + 6_000 draws total) took 483 seconds.


  Convergence: max R-hat=1.0073
        Colour  posterior_mean   hdi_2.5  hdi_97.5
         green        0.294091 -1.402241  2.017228
          gold        0.203526 -1.495520  1.916799
        hybrid        0.196867 -1.505340  1.921171
closed/unknown        0.155258 -1.544984  1.875731
        bronze        0.126888 -1.570635  1.844397

  Pairwise posterior probabilities:
    P(gold > green) = 0.0000
    P(gold > hybrid) = 0.6232
    P(gold > bronze) = 0.9995
    P(green > hybrid) = 0.9998
    P(green > bronze) = 1.0000
    P(hybrid > bronze) = 0.9867
  NOTE: idata.to_netcdf() failed (cannot write NetCDF files with format='NETCDF4' because none of the suitable backend libraries (netCDF4, h5netcdf) are installed); no netCDF4/h5netcdf backend is installed. Falling back to pickle: outputs/colour_heterogeneity/tables/C4_idata.pkl
  For standard .nc output next time, run:
    pip install netCDF4 --break-system-packages
  (or: pip install h5netcdf --break-system-packages)
  C4_colour_posteri

In [4]:
#

In [12]:
"""
Section 4.6 Figures — "Governance-Intensity Heterogeneity by OA Colour"
==========================================================================
Purpose
-------
Visualizes the HONEST re-framing of the OA-colour heterogeneity results:

  - P1 (gold > green level effect) is REJECTED — green shows the larger
    and more precisely estimated premium across every method (C3 OLS,
    C4 Bayesian posterior, C5 spline, C6 superstar odds).
  - The data instead support a two-axis story:
      Axis 1 (ADOPTION):  which colour institutions can most directly
                           compel -> gold share rises +29.7pp post-2015,
                           tracking the national mandate almost exactly.
      Axis 2 (VALUE):     which colour actually drives the patent-citation
                           premium -> green, plausibly because green
                           papers already carry higher academic-citation
                           signal (C2 balance table) and connect to the
                           32.3% academic-citation mediation channel
                           already established in Section 4.4.
  - This still gives the orthogonal-platform framework a prediction the
    pure diffusion/visibility-shock account cannot generate (colours
    should NOT differ at all under equal-readability diffusion), while
    not overstating what the data show.

This script assumes you have already run:
    oa_pipeline.py                          -> outputs/tables/analysis_final.csv
    oa_colour_heterogeneity_pipeline.py      -> outputs/colour_heterogeneity/...

It reads directly from those output files rather than re-fitting any
models, so it is fast and reproducible from the existing run.

Outputs (all written to OUT_DIR)
---------------------------------
  fig11_adoption_vs_value_decoupling.png   (2-panel: composition shift +
                                             Bayesian posterior densities)
  fig12_honest_colour_forest.png           (2-panel forest: level effect
                                             + penetration interaction,
                                             P1/P2 verdict annotated directly
                                             on the plot)
  fig13_selection_diagnostic.png           (academic-citation balance by
                                             colour -> visualizes the
                                             selection concern that
                                             motivates the "value axis =
                                             green" reinterpretation)
  table_4_6_summary.csv                    (one combined table for the
                                             manuscript / response letter)

Usage
-----
    python fig_4_6_governance_reframing.py \
        --colour-dir outputs/colour_heterogeneity \
        --analysis-csv outputs/tables/analysis_final.csv \
        --out-dir outputs/figures/07_combined
"""

import argparse
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ==============================================================================
# Style (matches the main pipeline's plot conventions)
# ==============================================================================
COLOUR_ORDER = ["closed/unknown", "bronze", "green", "hybrid", "gold"]
COLOUR_PALETTE = {
    "closed/unknown": "#7F8C8D", "bronze": "#B08D57", "green": "#1A7A4A",
    "hybrid": "#7D3C98", "gold": "#D4A017",
}
C_BLUE, C_RED, C_TEAL, C_GREY, C_ORANGE = (
    "#1D4E89", "#C0392B", "#148F77", "#7F8C8D", "#E67E22"
)
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
    "figure.dpi": 150, "savefig.dpi": 200, "savefig.bbox": "tight",
})


def _load_json(path: Path) -> dict:
    with open(path) as f:
        return json.load(f)


# ==============================================================================
# FIGURE 11 — Adoption axis vs Value axis decoupling
# ==============================================================================
def fig11_adoption_vs_value(colour_dir: Path, out_dir: Path):
    """
    Left panel:  stacked composition (from C1) showing gold's mandate-driven
                 rise -> the ADOPTION axis.
    Right panel: Bayesian posterior densities (from C4) showing green's
                 larger, more precisely estimated effect -> the VALUE axis.
                 Annotated with the pairwise posterior probability
                 P(green > gold) = 1.0000 (computed as 1 - P(gold>green)
                 = 1 - 0.0000 from the C4 run).
    """
    print("[Fig 11] Adoption vs Value decoupling")

    yr_colour = pd.read_csv(colour_dir / "tables" / "C1_colour_share_by_year.csv",
                             index_col=0)
    yr_colour = yr_colour.reindex(columns=COLOUR_ORDER, fill_value=0)

    c4_json = _load_json(colour_dir / "tables" / "C4_bayesian_colour_hierarchy.json")
    pairwise = c4_json.get("pairwise_probabilities", {})
    p_gold_gt_green = pairwise.get("P(gold > green)", None)

    # Try to recover raw posterior draws for a proper KDE; fall back to a
    # Normal approximation from the summary table if the pickle is unavailable
    # (netCDF backend was missing in the original run -> pickle fallback).
    idata = None
    for cand in ["C4_idata.nc", "C4_idata.pkl"]:
        p = colour_dir / "tables" / cand
        if p.exists():
            if cand.endswith(".pkl"):
                with open(p, "rb") as f:
                    idata = pickle.load(f)
            else:
                import arviz as az
                idata = az.from_netcdf(str(p))
            break

    summ = pd.read_csv(colour_dir / "tables" / "C4_colour_posterior_summary.csv")
    summ = summ.set_index("Colour").reindex(
        ["gold", "green", "hybrid", "bronze", "closed/unknown"]).dropna(how="all")

    fig = plt.figure(figsize=(15, 6))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1.15, 1], wspace=0.28)

    # -- Left: composition shift (ADOPTION axis) --------------------------------
    ax0 = fig.add_subplot(gs[0])
    bottom = np.zeros(len(yr_colour))
    years = yr_colour.index.values
    for col in COLOUR_ORDER:
        ax0.bar(years, yr_colour[col].values, bottom=bottom,
                color=COLOUR_PALETTE[col], label=col, width=0.85,
                edgecolor="white", linewidth=0.3)
        bottom += yr_colour[col].values
    ax0.axvline(2015, color="black", lw=1.6, ls="--", alpha=0.75)
    ax0.text(2015.3, 95, "2015 national\nOA mandate", fontsize=9, va="top")
    ax0.annotate("Gold share:\n+29.7 pp\npost-2015",
                 xy=(2020, 70), fontsize=10, fontweight="bold",
                 color=COLOUR_PALETTE["gold"],
                 bbox=dict(boxstyle="round", fc="white", ec=COLOUR_PALETTE["gold"]))
    ax0.set_ylabel("Share of papers (%)")
    ax0.set_xlabel("Publication Year")
    ax0.set_title("A. Adoption Axis: Gold OA Tracks the Institutional Mandate")
    ax0.legend(loc="upper left", ncol=2, fontsize=8.5, framealpha=0.95)
    ax0.set_ylim(0, 105)

    # -- Right: Bayesian posteriors (VALUE axis) --------------------------------
    ax1 = fig.add_subplot(gs[1])
    if idata is not None:
        colours_in_model = None
        try:
            post = idata.posterior["colour_effect"]
            colours_in_model = list(post.coords.get("colour_effect_dim_0", range(post.shape[-1])))
        except Exception:
            post = None
        if post is not None:
            arr = post.values.reshape(-1, post.shape[-1])
            # Column order follows np.unique() alphabetical order used when fitting:
            # ['bronze', 'closed/unknown', 'gold', 'green', 'hybrid']
            fitted_order = ["bronze", "closed/unknown", "gold", "green", "hybrid"]
            for i, c in enumerate(fitted_order):
                if c in ("gold", "green", "hybrid", "bronze"):
                    sns.kdeplot(arr[:, i], ax=ax1, color=COLOUR_PALETTE[c],
                                lw=2.4, label=c, fill=True, alpha=0.15)
    else:
        # Normal-approximation fallback from the summary CSV (mean + HDI width)
        for c in ["gold", "green", "hybrid", "bronze"]:
            if c not in summ.index:
                continue
            m = summ.loc[c, "posterior_mean"]
            sd = (summ.loc[c, "hdi_97.5"] - summ.loc[c, "hdi_2.5"]) / (2 * 1.96)
            x = np.linspace(m - 4 * sd, m + 4 * sd, 300)
            y = np.exp(-0.5 * ((x - m) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
            ax1.plot(x, y, color=COLOUR_PALETTE[c], lw=2.4, label=c)
            ax1.fill_between(x, y, color=COLOUR_PALETTE[c], alpha=0.15)

    ax1.axvline(0, color="black", lw=1, ls=":")
    ax1.set_xlabel("Posterior colour effect on log(1+patent citations)")
    ax1.set_ylabel("Density")
    ax1.set_title("B. Value Axis: Green Shows the Larger, Sharper Posterior")
    ax1.legend(fontsize=9)
    if p_gold_gt_green is not None:
        ax1.annotate(f"P(green > gold) = {1 - p_gold_gt_green:.4f}",
                     xy=(0.97, 0.92), xycoords="axes fraction", ha="right",
                     fontsize=10, fontweight="bold",
                     bbox=dict(boxstyle="round", fc="white", ec=C_GREY))

    fig.suptitle("Figure 11. Decoupling the Adoption and Value Axes of OA Governance",
                  fontsize=14, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.savefig(out_dir / "fig11_adoption_vs_value_decoupling.png")
    plt.close()
    print("  fig11_adoption_vs_value_decoupling.png written")


# ==============================================================================
# FIGURE 12 — Honest forest plot: level effect + penetration interaction
# ==============================================================================
def fig12_honest_forest(colour_dir: Path, out_dir: Path):
    """
    Side-by-side forest plots directly labelled with the P1/P2 verdicts,
    rather than letting the reader infer support/non-support from raw
    coefficients. This is the figure that should accompany the text
    stating P1 is rejected and P2 is only weakly/directionally supported.
    """
    print("[Fig 12] Honest colour-effect forest plot")
    coef = pd.read_csv(colour_dir / "tables" / "C3_colour_regression_coefs.csv")
    coef["Colour"] = pd.Categorical(coef["Colour"],
                                     categories=["gold", "green", "hybrid", "bronze"],
                                     ordered=True)
    coef = coef.sort_values("Colour")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    # -- Panel A: level effects --------------------------------------------------
    ax = axes[0]
    for _, row in coef.iterrows():
        c = row["Colour"]
        sig = row["Level_p"] < 0.05
        ax.errorbar(row["Level_effect"], c, fmt="o" if sig else "o",
                    color=COLOUR_PALETTE.get(c, C_GREY),
                    ms=12 if sig else 9,
                    mfc=COLOUR_PALETTE.get(c, C_GREY) if sig else "white",
                    mec=COLOUR_PALETTE.get(c, C_GREY), mew=1.8,
                    capsize=4)
    ax.axvline(0, color="black", lw=1)
    ax.set_xlabel("Level effect vs. closed/unknown (log points)")
    ax.set_title("A. Level Effect — P1 (gold > green) NOT SUPPORTED",
                  fontsize=11.5, color=C_RED)
    ax.text(0.02, -0.28, "Filled = p<.05 · Open = n.s.", transform=ax.transAxes,
            fontsize=8.5, style="italic")

    # -- Panel B: penetration interaction ----------------------------------------
    ax = axes[1]
    for _, row in coef.iterrows():
        c = row["Colour"]
        sig = row["Penetration_interaction_p"] < 0.05
        ax.errorbar(row["Penetration_interaction"], c, fmt="o",
                    color=COLOUR_PALETTE.get(c, C_GREY),
                    ms=12 if sig else 9,
                    mfc=COLOUR_PALETTE.get(c, C_GREY) if sig else "white",
                    mec=COLOUR_PALETTE.get(c, C_GREY), mew=1.8,
                    capsize=4)
    ax.axvline(0, color="black", lw=1)
    ax.set_xlabel("Colour × pre-field-OA-penetration interaction")
    ax.set_title("B. Penetration Interaction — P2 (gold fastest) WEAK,\n"
                  "sign runs counter to H2 attenuation logic",
                  fontsize=11.5, color=C_ORANGE)
    ax.text(0.02, -0.32, "Filled = p<.05 · Open = n.s.\n"
            "Note: positive sign = premium RISES with penetration\n"
            "for every colour except closed/unknown reference",
            transform=ax.transAxes, fontsize=8, style="italic")

    fig.suptitle("Figure 12. Colour-Specific Effects with Explicit Hypothesis Verdicts",
                  fontsize=14, fontweight="bold", y=1.06)
    plt.tight_layout()
    plt.savefig(out_dir / "fig12_honest_colour_forest.png")
    plt.close()
    print("  fig12_honest_colour_forest.png written")


# ==============================================================================
# FIGURE 13 — Selection diagnostic (why green's premium may not be governance)
# ==============================================================================
def fig13_selection_diagnostic(analysis_csv: Path, out_dir: Path):
    """
    Visualizes the C2 balance-table concern directly: green papers already
    carry a higher academic-citation signal pre-treatment, which is the
    same channel identified as the 32.3% mediation pathway in Section 4.4.
    This figure is what lets the manuscript say "green's advantage is
    plausibly value-channel-mediated rather than governance-intensity-
    driven" without over-claiming a causal test the design cannot deliver.
    """
    print("[Fig 13] Selection diagnostic: academic citation balance by colour")
    df = pd.read_csv(analysis_csv)
    if "OA_Colour" not in df.columns or "log_citation" not in df.columns:
        print("  WARNING: required columns not found in analysis_final.csv -- skipping.")
        return

    d = df.copy()
    d["OA_Colour"] = d["OA_Colour"].fillna("closed/unknown").astype(str).str.lower()
    d["OA_Colour"] = d["OA_Colour"].replace({"unknown": "closed/unknown"})
    d = d[d["OA_Colour"].isin(COLOUR_ORDER)]
    d["OA_Colour"] = pd.Categorical(d["OA_Colour"], categories=COLOUR_ORDER, ordered=True)

    d = d.dropna(subset=["log_citation"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 7.2))

    # -- Panel A: violin of academic citation by colour --------------------------
    ax = axes[0]
    sns.violinplot(data=d, x="OA_Colour", y="log_citation", order=COLOUR_ORDER,
                    palette=COLOUR_PALETTE, ax=ax, cut=0, inner="quartile")
    ax.set_xlabel("")
    ax.set_ylabel("log(1 + academic citations)")
    ax.set_title("A. Green Papers Already Carry Higher Academic\nVisibility Pre-Treatment",
                  pad=14)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right",
                        rotation_mode="anchor")
    ax.tick_params(axis="x", pad=2)

    # -- Panel B: mediation-channel schematic (annotated arrow diagram) ---------
    ax = axes[1]
    ax.axis("off")
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)

    box_style = dict(boxstyle="round,pad=0.6", ec="black", lw=1.4)
    Y_BOXES = 8.5             # top row of boxes
    Y_ARROW_LABEL_TOP = 9.2
    Y_DIRECT_ARROW = 1.0      # direct-path arrow drawn near the bottom
    Y_DIRECT_LABEL = 0.25

    ax.text(1.2, Y_BOXES, "OA Colour\n(green)", ha="center", va="center", fontsize=11,
            bbox=dict(fc=COLOUR_PALETTE["green"], alpha=0.25, **box_style))
    ax.text(5.0, Y_BOXES, "Academic\nCitations", ha="center", va="center", fontsize=11,
            bbox=dict(fc="white", **box_style))
    ax.text(8.8, Y_BOXES, "Patent\nCitations", ha="center", va="center", fontsize=11,
            bbox=dict(fc="white", **box_style))

    ax.annotate("", xy=(4.0, Y_BOXES), xytext=(2.05, Y_BOXES),
                arrowprops=dict(arrowstyle="->", lw=2.0, color=C_TEAL))
    ax.annotate("", xy=(7.8, Y_BOXES), xytext=(6.0, Y_BOXES),
                arrowprops=dict(arrowstyle="->", lw=2.0, color=C_TEAL))
    ax.text(3.0, Y_ARROW_LABEL_TOP, "elevated at\nbaseline (C2)", ha="center",
            va="bottom", fontsize=9, color=C_TEAL, style="italic")
    ax.text(6.9, Y_ARROW_LABEL_TOP, "32.3% mediation\nshare (\u00a74.4)", ha="center",
            va="bottom", fontsize=9, color=C_TEAL, style="italic")

    ax.annotate("", xy=(8.8, Y_DIRECT_ARROW), xytext=(1.2, Y_DIRECT_ARROW),
                arrowprops=dict(arrowstyle="->", lw=1.4, color=C_GREY, ls="--"))
    ax.text(5, Y_DIRECT_LABEL, "direct path (67.7%, \u00a74.4)", ha="center",
            va="bottom", fontsize=9, color=C_GREY, style="italic")

    # vertical connector showing the two paths both run from colour to citations
    ax.annotate("", xy=(1.2, Y_DIRECT_ARROW + 0.5), xytext=(1.2, Y_BOXES - 0.55),
                arrowprops=dict(arrowstyle="-", lw=1.2, color=C_GREY, ls=":"))
    ax.annotate("", xy=(8.8, Y_DIRECT_ARROW + 0.5), xytext=(8.8, Y_BOXES - 0.55),
                arrowprops=dict(arrowstyle="-", lw=1.2, color=C_GREY, ls=":"))

    ax.set_title("B. OA Colour \u2192 Academic Citations \u2192 Patent\n"
                  "Citations Pathway (\u00a74.4)", pad=14)

    fig.suptitle("Figure 13. Selection Diagnostic and Mediation Pathway",
                  fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.subplots_adjust(wspace=0.32, bottom=0.16)
    plt.savefig(out_dir / "fig13_selection_diagnostic.png")
    plt.close()
    print("  fig13_selection_diagnostic.png written")


# ==============================================================================
# Combined summary table for the manuscript / response letter
# ==============================================================================
def build_summary_table(colour_dir: Path, out_dir: Path):
    print("[Table] Combined Section 4.6 summary")
    coef = pd.read_csv(colour_dir / "tables" / "C3_colour_regression_coefs.csv")
    bayes = pd.read_csv(colour_dir / "tables" / "C4_colour_posterior_summary.csv")
    superstar = pd.read_csv(colour_dir / "tables" / "C6_superstar_odds_by_colour.csv")

    merged = (coef.merge(bayes.rename(columns={"Colour": "Colour"}), on="Colour", how="left")
                    .merge(superstar.rename(columns={"Colour": "Colour"}), on="Colour", how="left"))
    merged = merged[merged["Colour"].isin(["gold", "green", "hybrid", "bronze"])]
    merged = merged.sort_values("posterior_mean", ascending=False)

    keep_cols = ["Colour", "Level_effect", "Level_p", "posterior_mean",
                 "hdi_2.5", "hdi_97.5", "Penetration_interaction",
                 "Penetration_interaction_p", "Odds_Ratio", "p_value"]
    keep_cols = [c for c in keep_cols if c in merged.columns]
    merged[keep_cols].round(4).to_csv(out_dir / "table_4_6_summary.csv", index=False)
    print(merged[keep_cols].round(4).to_string(index=False))
    print("  table_4_6_summary.csv written")


# ==============================================================================
# MAIN
# ==============================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--colour-dir", type=str, default="outputs/colour_heterogeneity")
    ap.add_argument("--analysis-csv", type=str, default="outputs/tables/analysis_final.csv")
    ap.add_argument("--out-dir", type=str, default="outputs/figures/07_combined")
    args, _unknown = ap.parse_known_args()  # tolerate Jupyter's own -f kernel arg

    colour_dir = Path(args.colour_dir)
    analysis_csv = Path(args.analysis_csv)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 70)
    print("  Section 4.6 Figures — Governance-Intensity Heterogeneity (honest)")
    print("=" * 70)

    fig11_adoption_vs_value(colour_dir, out_dir)
    fig12_honest_forest(colour_dir, out_dir)
    fig13_selection_diagnostic(analysis_csv, out_dir)
    build_summary_table(colour_dir, out_dir)

    print("\nDone -> " + str(out_dir))
    print("""
  Manuscript integration note
  ----------------------------
  Fig. 11 replaces any single "gold > green" claim with the adoption/value
  decoupling story. Fig. 12 is designed to be read alongside text that
  explicitly states P1 is rejected. Fig. 13 supplies the mediation-based
  reinterpretation so the section reads as a genuine extension of §4.4
  rather than a standalone (and currently unsupported) governance-ranking
  claim.
    """)


if __name__ == "__main__":
    main()


  Section 4.6 Figures — Governance-Intensity Heterogeneity (honest)
[Fig 11] Adoption vs Value decoupling
  fig11_adoption_vs_value_decoupling.png written
[Fig 12] Honest colour-effect forest plot
  fig12_honest_colour_forest.png written
[Fig 13] Selection diagnostic: academic citation balance by colour
  fig13_selection_diagnostic.png written
[Table] Combined Section 4.6 summary
Colour  Level_effect  Level_p  posterior_mean  hdi_2.5  hdi_97.5  Penetration_interaction  Penetration_interaction_p  Odds_Ratio  p_value
 green        0.0443   0.3183          0.2941  -1.4022    2.0172                   0.1860                     0.0061      2.6043   0.0003
  gold        0.0253   0.3698          0.2035  -1.4955    1.9168                   0.0580                     0.2038      1.5664   0.0621
hybrid       -0.0218   0.7764          0.1969  -1.5053    1.9212                   0.1189                     0.0812      1.8592   0.1153
bronze       -0.1203   0.1532          0.1269  -1.5706    1.8444